# Pilot vs Transformer — T4 ×2 · four hours
Research release: Pilot (formerly Matrix EXILRM). The embedded code is the exact source from the reported run. Internal identifiers `matrix_fixed` and `standard` are retained for reproducibility.

Required Notice: Copyright (c) 2026 Rickey1990. Software: PolyForm Noncommercial 1.0.0; see LICENSE.


One independent model per GPU. Original fixed-decay EXILRM (0.9) versus the original standard four-head Transformer with learned positions. No changes to the nonlinear matrix equations, no memory supervision, no extra blocks. Same real FineWeb-Edu corpus, tokenizer, parameter matching, 256-token context and 16 sequences/update (4,096 targets).

**240-minute scheduling target** starting at setup. Up to 20 minutes for fresh data, 18 minutes for six calibration stages, 20 minutes for frozen evaluation and 5 minutes for termination/export. The remaining training time uses a 75% safety factor. Select the fastest numerically verified backend on each GPU; both models receive identical target counts. Compare the highest shared checkpoint if a worker stops early. Not a hard OS-level guarantee; incomplete optional tests are labelled.

Three equivalent history algorithms: literal sequential scan, zero-padded FFT and associative parallel prefix scan (doubling, O(log T) stages, O(T log T) work). The gated nonlinear matrix recurrence remains sequential. CUDA graphs replay the same operations; no approximate recurrence or precision change. Before accepting any candidate, compare two full real-batch forward/backward/AdamW steps to eager FFT with identical initial weights. Independently check history values and gradients against direct float64 summation at multiple lengths and decays. Record failures in worker logs; a failed candidate cannot be selected. After training, compare FFT, sequential and parallel scan on the same checkpoint. GPU performance is measured, not assumed.

Kaggle: select T4 x2, attach FineWeb-Edu input or reuse prepared data, Run All. Single seed, exploratory results. The 100M cap is not a promise of throughput. With 177 minutes remaining and 75% margin, 333 / 1,000 / 2,000 / 5,000 tokens/s means about 2.65M / 7.97M / 15.93M / 39.83M targets per model, before data cap and rounding.


In [ ]:
# Start the complete notebook clock before imports or dependency checks.
import time, os, sys, subprocess, importlib.util
from pathlib import Path
RUN_STARTED = time.time()
WALL_MINUTES = 240
DEADLINE = RUN_STARTED + WALL_MINUTES * 60
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
# PyTorch must be supplied by the Kaggle GPU image. Only small missing data dependencies are installed.
missing = [x for x in ('sentencepiece','pyarrow','numpy') if importlib.util.find_spec(x) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing], timeout=180)
import torch
print('Visible GPUs:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2 (two visible devices).'
print('Wall budget:',WALL_MINUTES,'minutes')

## 1 · Embedded implementation
These two cells write the original research module and the time-budgeted runner. No separate source download is necessary. The original equations are preserved; the new runner changes scheduling, context/batch configuration, precision policy and evaluation cadence. CUDA graph capture is checked against eager loss and gradients before training, and used only when faster. Eager is the fallback during calibration.

In [ ]:
(WORK/'research500.py').write_text('"""FineWeb-Edu 500M Matrix EXILRM experiment.\nEmbedded in the standalone Kaggle notebook; Python 3.10+, torch, numpy,\nSentencePiece, Hugging Face datasets, pandas (optional for plotting).\n\nAll targets are scored exactly once per model within a static, frozen training\ncorpus. Dataset is sourced from locally attached FineWeb-Edu Parquet sample. Neither\nvalidation nor test is used to train the tokenizer or model.\n"""\nfrom __future__ import annotations\nimport os, sys, json, csv, math, time, copy, hashlib, random, signal, shutil, zipfile, traceback, platform, warnings, functools\nfrom pathlib import Path\nfrom dataclasses import dataclass, asdict\nfrom collections import defaultdict\nfrom contextlib import nullcontext\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nVERSION = \'exilrm-500m-v1\'\nALL_MODELS = (\'standard\',\'matrix_fixed\',\'matrix_learned\',\'matrix_multi\',\'matrix_readnorm\')\nPREFLIGHT_DOCS = 1000\nSTOP_REQUESTED = False\n\n\ndef request_stop(signum, frame):\n    global STOP_REQUESTED\n    STOP_REQUESTED = True\n    print(\'STOP REQUESTED: finishing current batch and saving a resumable checkpoint\', flush=True)\n\n\ndef atomic_json(path, obj):\n    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)\n    q = p.with_name(p.name + \'.tmp\')\n    q.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str), encoding=\'utf8\')\n    q.replace(p)\n\n\ndef read_json(path, default=None):\n    p=Path(path)\n    if not p.exists(): return default\n    return json.loads(p.read_text(encoding=\'utf8\'))\n\n\ndef append_csv(path, record):\n    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)\n    exists=p.exists() and p.stat().st_size>0\n    with p.open(\'a\',newline=\'\',encoding=\'utf8\') as f:\n        w=csv.DictWriter(f,fieldnames=list(record),extrasaction=\'ignore\')\n        if not exists: w.writeheader()\n        w.writerow(record)\n\n\ndef sha256_file(path, chunk=8*1024*1024):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        while True:\n            b=f.read(chunk)\n            if not b: break\n            h.update(b)\n    return h.hexdigest()\n\n\ndef seed_everything(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\n@dataclass\nclass Config:\n    root: str = \'/kaggle/working/exilrm_500m_experiment\'\n    resume_input_root: str = \'\'\n    dataset_id: str = \'HuggingFaceFW/fineweb-edu\'\n    dataset_subset: str = \'sample-10BT\'\n    dataset_revision: str = \'main\'  # freeze to resolved commit SHA at collection time\n    local_parquet_glob: str = \'\'  # e.g. /kaggle/input/**/000_00000.parquet; no remote streaming\n    scored_tokens: int = 500_000_000\n    valid_tokens: int = 2_000_000\n    test_tokens: int = 2_000_000\n    tokenizer_sample_chars: int = 20_000_000\n    vocab: int = 4096\n    context: int = 512\n    width: int = 64\n    hidden: int = 256\n    layers: int = 3\n    heads: int = 4\n    batch: int = 8\n    matrix_heads: int = 8\n    history_backend: str = "fft"\n    window_examples: int = 8\n    compile_chunks: bool = False\n    seed: int = 20261109\n    max_lr: float = 0.00075\n    min_lr: float = 0.00005\n    warmup_tokens: int = 4_000_000\n    weight_decay: float = 0.01\n    grad_clip: float = 1.0\n    use_amp: bool = True\n    checkpoint_every: int = 1_000_000\n    archive_every: int = 25_000_000\n    fast_eval_every: int = 1_000_000\n    probe_every: int = 10_000_000\n    full_eval_every: int = 25_000_000\n    models: tuple = ALL_MODELS\n    max_session_minutes: int = 600\n    max_milestones_per_session: int = 0  # zero = time-limited only\n    report_zip_limit_mb: int = 300  # strictly <350 MB including overhead\n    auto_stop_on_nan: bool = True\n    stop_on_two_severe_val_regressions: bool = True\n    smoke: bool = False\n    # Optional maximum minutes checked at optimizer boundaries\n\n\n    def check(self):\n        if self.width % self.matrix_heads: raise ValueError("width must divide matrix_heads")\n        if self.history_backend not in ("fft","scan","parallel"): raise ValueError("bad history backend")\n        if self.width % self.heads: raise ValueError(\'width must divide heads\')\n        if not 0 < self.vocab <= 65535: raise ValueError(\'uint16 requires vocab<=65535\')\n        if self.context < 32: raise ValueError(\'context too small\')\n        if self.scored_tokens < 1: raise ValueError(\'scored_tokens must be >0\')\n        if self.checkpoint_every < 1: raise ValueError(\'checkpoint_every must be >0\')\n        if not self.models or not set(self.models).issubset(ALL_MODELS):\n            raise ValueError(\'Unknown model in \'+str(self.models))\n        if self.report_zip_limit_mb >= 350: raise ValueError(\'ZIP limit must be <350 MB\')\n        return self\n\n\ndef freeze_config(cfg):\n    out=Path(cfg.root);out.mkdir(parents=True,exist_ok=True)\n    frozen=json.loads(json.dumps(asdict(cfg)))  # normalize tuple/list for JSON resume\n    for key in (\'max_session_minutes\',\'max_milestones_per_session\',\'resume_input_root\',\'compile_chunks\'):\n        frozen.pop(key,None)\n    p=out/\'experiment_config.json\'\n    old=read_json(p)\n    if old is None: atomic_json(p,frozen)\n    elif old != frozen:\n        changes={k:(old.get(k),frozen.get(k)) for k in sorted(set(old)|set(frozen)) if old.get(k)!=frozen.get(k)}\n        raise RuntimeError(\'Experiment configuration differs from previous run. Start a NEW root for a new experiment. Changes: \'+str(changes))\n\n\ndef local_parquet_paths(cfg):\n    if not cfg.local_parquet_glob:\n        return []\n    import glob\n    paths = [Path(p) for p in sorted(glob.glob(cfg.local_parquet_glob, recursive=True)) if Path(p).is_file()]\n    if not paths:\n        raise FileNotFoundError(\'No local Parquet data found for: \' + cfg.local_parquet_glob +\n            \'. In Kaggle: Add Input > Upload a Dataset containing 000_00000.parquet. \' +\n            \'Check the actual file path under /kaggle/input and adjust local_parquet_glob.\')\n    return paths\n\n\ndef source_iterator(cfg, revision=None):\n    if cfg.local_parquet_glob:\n        import pyarrow.parquet as pq\n        for path in local_parquet_paths(cfg):\n            for batch in pq.ParquetFile(path).iter_batches(batch_size=1024,columns=[\'text\']):\n                for txt in batch.column(0).to_pylist():\n                    if isinstance(txt,str) and txt.strip():yield txt\n    else:\n        from datasets import load_dataset\n        ds=load_dataset(cfg.dataset_id,name=cfg.dataset_subset,split=\'train\',revision=revision or cfg.dataset_revision,streaming=True)\n        for item in ds:\n            txt=item.get(\'text\',\'\')\n            if isinstance(txt,str) and txt.strip():yield txt\n\n\ndef resolved_dataset_revision(cfg):\n    if cfg.local_parquet_glob:\n        paths=local_parquet_paths(cfg)\n        # Source fingerprint used for resume integrity; file hashes are saved in the data manifest.\n        digests = [{\'name\':p.name,\'size\':p.stat().st_size,\'sha256\':sha256_file(p)} for p in paths]\n        data=json.dumps(digests,sort_keys=True).encode(\'utf8\')\n        pinned=\'local-parquet:\'+hashlib.sha256(data).hexdigest()\n        atomic_json(Path(cfg.root)/\'data\'/\'local_source_manifest.json\',{\'files\':digests,\'revision\':pinned})\n        print(\'Local Parquet input verified:\', \', \'.join(p.name for p in paths),flush=True)\n        return pinned\n    if cfg.dataset_revision != \'main\': return cfg.dataset_revision\n    try:\n        from huggingface_hub import HfApi\n        return HfApi().dataset_info(cfg.dataset_id).sha\n    except Exception as e:\n        print(\'WARNING: dataset commit could not be pinned:\',str(e)[:160],flush=True)\n        return \'main\'\n\n\ndef prep_tokenizer(cfg, out, revision):\n    import sentencepiece as spm\n    dir=out/\'data\';dir.mkdir(parents=True,exist_ok=True)\n    path=dir/\'spm.model\'\n    if path.exists():\n        sp=spm.SentencePieceProcessor(model_file=str(path))\n        if sp.vocab_size() > 65535: raise ValueError(\'Tokenizer too large for uint16 storage\')\n        return sp\n    sample=dir/\'tokenizer_train.txt\'\n    if not sample.exists():\n        print(\'Collecting tokenizer-training text from the train hash split...\',flush=True)\n        count=0; docs=0\n        partial=sample.with_suffix(\'.txt.part\')\n        with partial.open(\'w\',encoding=\'utf8\') as f:\n            for text in source_iterator(cfg,revision):\n                if split_for_text(text)!=\'train\':continue\n                text=text.replace(\'\\0\',\' \').strip()\n                if len(text)>12000:text=text[:12000]\n                if not text:continue\n                f.write(\' \'.join(text.split())+\'\\n\')\n                count+=len(text);docs+=1\n                if count>=cfg.tokenizer_sample_chars:break\n        if not count: raise RuntimeError(\'Tokenizer training sample is empty\')\n        partial.replace(sample)\n        atomic_json(dir/\'tokenizer_sample_manifest.json\',{\'source\':cfg.dataset_id,\'subset\':cfg.dataset_subset,\n             \'revision\':revision,\'train_docs\':docs,\'characters\':count})\n    print(\'Training BPE tokenizer...\',flush=True)\n    spm.SentencePieceTrainer.Train(input=str(sample),model_prefix=str(dir/\'spm\'),\n        vocab_size=cfg.vocab, model_type=\'bpe\',character_coverage=0.9995,\n        bos_id=-1,eos_id=1,unk_id=0,pad_id=-1,hard_vocab_limit=False,\n        num_threads=4,max_sentence_length=16384)\n    sample.unlink(missing_ok=True)\n    sp=spm.SentencePieceProcessor(model_file=str(path))\n    print(\'Tokenizer vocab:\',sp.vocab_size(),flush=True)\n    return sp\n\n\ndef split_for_text(text):\n    # Exact duplicate documents assigned to same split; near duplicates remain a caveat.\n    h=int.from_bytes(hashlib.sha256(text.encode(\'utf8\',\'ignore\')).digest()[:8],\'big\') % 1000\n    return \'valid\' if h<5 else (\'test\' if h<10 else \'train\')\n\n\ndef create_smoke_corpus(cfg):\n    """Offline synthetic data ONLY for implementation tests; never scientific results."""\n    import sentencepiece as spm\n    out=Path(cfg.root); d=out/\'data\';d.mkdir(parents=True,exist_ok=True)\n    txt=d/\'mock.txt\'\n    rng=random.Random(1005)\n    words=(\'Alice Bob Carol David remembered blue green red gold mapping story river \'+\n           \'the bright dog walked home after lunch a small question about science and books\').split()\n    with txt.open(\'w\') as f:\n        for i in range(1800):\n            f.write(\' \'.join(rng.choices(words,k=38))+\'\\n\')\n    spm.SentencePieceTrainer.Train(input=str(txt),model_prefix=str(d/\'spm\'),\n        vocab_size=min(cfg.vocab,256),model_type=\'bpe\',bos_id=-1,eos_id=1,\n        unk_id=0,pad_id=-1,hard_vocab_limit=False, num_threads=2)\n    sp=spm.SentencePieceProcessor(model_file=str(d/\'spm.model\'))\n    vocab=sp.vocab_size();rng=np.random.default_rng(149)\n    lengths={\'train\':cfg.scored_tokens+1,\'valid\':cfg.valid_tokens,\'test\':cfg.test_tokens}\n    for split,n in lengths.items():\n        p=d/f\'{split}.bin\'\n        arr=np.memmap(p,dtype=np.uint16,mode=\'w+\',shape=(n,))\n        arr[:]=rng.integers(2,vocab,size=n,dtype=np.uint16);arr.flush();del arr\n    manifest={\'source\':\'SYNTHETIC_SMOKE_ONLY\',\'completed\':True,\'revision\':\'none\',\'sizes\':lengths,\n              \'sha256_tokenizer\':sha256_file(d/\'spm.model\'),\'vocab\':vocab,\'documents_seen\':0,\n              \'sha256_token_files\':{split:sha256_file(d/f\'{split}.bin\') for split in lengths}}\n    atomic_json(d/\'manifest.json\',manifest)\n    txt.unlink(missing_ok=True)\n    return manifest\n\n\ndef prepare_corpus(cfg):\n    out=Path(cfg.root); out.mkdir(parents=True,exist_ok=True)\n    if cfg.smoke:\n        if not (out/\'data\'/\'manifest.json\').exists():return create_smoke_corpus(cfg)\n        return read_json(out/\'data\'/\'manifest.json\')\n    d=out/\'data\';d.mkdir(exist_ok=True)\n    p=d/\'manifest.json\'\n    manifest=read_json(p)\n    expected={\'train\':cfg.scored_tokens+1,\'valid\':cfg.valid_tokens,\'test\':cfg.test_tokens}\n    if manifest and manifest.get(\'completed\'):\n        if manifest[\'sizes\'] != expected: raise ValueError(\'Incompatible prepared corpus sizes\')\n        expected_hashes=manifest.get(\'sha256_token_files\',{})\n        for key,n in expected.items():\n            f=d/f\'{key}.bin\'\n            if not f.exists() or f.stat().st_size!=2*n:\n                raise RuntimeError(\'Prepared token file missing/wrong size: \'+str(f))\n            if expected_hashes and sha256_file(f)!=expected_hashes[key]:\n                raise RuntimeError(\'Token corpus fingerprint mismatch: \'+str(f))\n        if sha256_file(d/\'spm.model\')!=manifest[\'sha256_tokenizer\']:raise RuntimeError(\'Tokenizer fingerprint mismatch\')\n        print(\'Using fingerprint-verified prepared token corpus:\',p,flush=True)\n        return manifest\n    revision=(manifest or {}).get(\'revision\') or read_json(d/\'source_revision.json\',{}).get(\'revision\') or resolved_dataset_revision(cfg)\n    if cfg.local_parquet_glob and resolved_dataset_revision(cfg)!=revision:raise RuntimeError(\'Local input changed since partial corpus preparation\')\n    atomic_json(d/\'source_revision.json\', {\'dataset_id\':cfg.dataset_id,\'subset\':cfg.dataset_subset,\'revision\':revision})\n    sp=prep_tokenizer(cfg,out,revision)\n    if manifest is None:\n        manifest={\'source\':cfg.dataset_id,\'subset\':cfg.dataset_subset,\'revision\':revision,\n                  \'sizes\':expected,\'vocab\':sp.vocab_size(),\'sha256_tokenizer\':sha256_file(d/\'spm.model\'),\n                  \'filled\':{\'train\':0,\'valid\':0,\'test\':0},\n                  \'documents_seen\':0,\'documents_split\':{\'train\':0,\'valid\':0,\'test\':0},\n                  \'completed\':False, \'exact_duplicates_grouped_by_hash\':True}\n        atomic_json(p,manifest)\n    elif manifest[\'sizes\']!=expected or manifest[\'sha256_tokenizer\']!=sha256_file(d/\'spm.model\'):\n        raise RuntimeError(\'Data manifest/tokenizer changed; refusing unsafe resume\')\n    mm={}\n    for key,n in expected.items():\n        f=d/f\'{key}.bin\'\n        if not f.exists():\n            with open(f,\'wb\') as o:o.truncate(n*2)\n        if f.stat().st_size!=2*n: raise RuntimeError(\'Incompatible token cache file \'+str(f))\n        mm[key]=np.memmap(f,dtype=np.uint16,mode=\'r+\',shape=(n,))\n    filled=dict(manifest[\'filled\']);docs_seen=manifest[\'documents_seen\'];docs_split=dict(manifest[\'documents_split\'])\n    print(\'Reading FineWeb-Edu source and building a ~1GB token corpus (local Parquet if configured).\',flush=True)\n    gen=source_iterator(cfg,revision)\n    if docs_seen:\n        # Exact deterministic skip from same immutable stream revision.\n        print(\'Restoring at source document\',docs_seen,flush=True)\n        for _ in range(docs_seen):next(gen)\n    since_save=0; timer=time.monotonic()\n    try:\n        for text in gen:\n            docs_seen+=1\n            split=split_for_text(text)\n            docs_split[split]+=1\n            required=expected[split]-filled[split]\n            if required>0:\n                ids=sp.encode(text,out_type=int)\n                if ids:\n                    ids.append(sp.eos_id())\n                    ids=ids[:required]\n                    mm[split][filled[split]:filled[split]+len(ids)]=np.asarray(ids,dtype=np.uint16)\n                    filled[split]+=len(ids)\n            since_save+=1\n            if since_save>=2000 or STOP_REQUESTED:\n                for array in mm.values():array.flush()\n                manifest.update({\'filled\':filled,\'documents_seen\':docs_seen,\'documents_split\':docs_split})\n                atomic_json(p,manifest);since_save=0\n                if docs_seen%20000<2000:\n                    print(\'Tokenizer positions:\',{k:f\'{filled[k]:,}/{expected[k]:,}\' for k in expected},flush=True)\n                if STOP_REQUESTED:raise KeyboardInterrupt(\'Stopped during dataset preparation. Resume with same data dir.\')\n            if all(filled[k]>=expected[k] for k in expected): break\n    finally:\n        for arr in mm.values():arr.flush()\n        manifest.update({\'filled\':filled,\'documents_seen\':docs_seen,\'documents_split\':docs_split})\n        atomic_json(p,manifest)\n    if not all(filled[k]>=expected[k] for k in expected):\n        raise RuntimeError(\'Stream ended before all split budgets filled. \'+str(filled))\n    manifest[\'sha256_token_files\']={key:sha256_file(d/f\'{key}.bin\') for key in expected}\n    manifest[\'completed\']=True\n    atomic_json(p,manifest)\n    print(\'PREPARED:\',{k:f\'{v:,}\' for k,v in filled.items()},\'in seconds\',int(time.monotonic()-timer),flush=True)\n    return manifest\n\n\nclass MLP(nn.Module):\n    def __init__(self,d=64,h=256):\n        super().__init__();self.fc1=nn.Linear(d,h);self.fc2=nn.Linear(h,d)\n    def forward(self,x):return self.fc2(F.gelu(self.fc1(x)))\n\n\nclass Attention(nn.Module):\n    """Four independent heads computed concurrently; optional pooled summary as an EXTRA path."""\n    def __init__(self, d=64, heads=4, pooled=False):\n        super().__init__(); assert d%heads==0\n        self.d=d;self.heads=heads;self.dh=d//heads;self.pooled=pooled\n        self.qkv=nn.Linear(d,3*d);self.out=nn.Linear(d,d)\n        if pooled:\n            self.pool_query=nn.Linear(d,self.dh,bias=False)\n            self.pool_out=nn.Linear(self.dh,d)\n            self.pool_logit=nn.Parameter(torch.tensor(-1.5))\n            nn.init.normal_(self.pool_out.weight,std=0.015)\n            nn.init.zeros_(self.pool_out.bias)\n        self.pool_mode=\'normal\';self.capture_stats=False;self.last_entropy=None\n\n    def forward(self,x):\n        b,t,d=x.shape\n        qkv=self.qkv(x).reshape(b,t,3,self.heads,self.dh)\n        q,k,v=qkv.unbind(2);q,k,v=[a.transpose(1,2) for a in (q,k,v)]\n        heads=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0).transpose(1,2)\n        base=self.out(heads.reshape(b,t,d))\n        if not self.pooled or self.pool_mode==\'off\':return base\n        query=self.pool_query(x).unsqueeze(2)\n        weights=((heads*query).sum(-1)*(self.dh**-0.5)).float().softmax(-1).to(heads.dtype)\n        if self.pool_mode==\'uniform\':weights=torch.full_like(weights,1/self.heads)\n        if self.pool_mode==\'shifted\':weights=weights.roll(shifts=1,dims=-1)\n        summary=(heads*weights.unsqueeze(-1)).sum(2)\n        contribution=self.pool_logit.sigmoid()*self.pool_out(summary)\n        if self.capture_stats:\n            probs=weights.detach().float().clamp_min(1e-9)\n            self.last_entropy=float((-(probs*probs.log()).sum(-1).mean()/math.log(self.heads)).item())\n        return base+contribution\n\n\nclass Block(nn.Module):\n    def __init__(self,d,h,heads,pooled,own_ff):\n        super().__init__();self.norm_a=nn.LayerNorm(d);self.attn=Attention(d,heads,pooled)\n        self.norm_f=nn.LayerNorm(d);self.ff=MLP(d,h) if own_ff else None\n    def forward(self,x,shared_ff=None):\n        z=x+self.attn(self.norm_a(x));return z+(self.ff if self.ff is not None else shared_ff)(self.norm_f(z))\n\n\nclass ReferenceModel(nn.Module):\n    def __init__(self,cfg,variant,vocab_size=None):\n        super().__init__()\n        if variant not in ALL_MODELS:raise ValueError(variant)\n        self.variant=variant;self.context=cfg.context;self.width=cfg.width\n        pooled=variant.endswith(\'_pool\');shared=variant.startswith(\'shared\')\n        v=vocab_size or cfg.vocab\n        self.embed=nn.Embedding(v,cfg.width);self.pos=nn.Embedding(cfg.context,cfg.width)\n        self.shared_ff=MLP(cfg.width,cfg.hidden) if shared else None\n        self.blocks=nn.ModuleList([Block(cfg.width,cfg.hidden,cfg.heads,pooled,not shared) for _ in range(cfg.layers)])\n        self.final_norm=nn.LayerNorm(cfg.width)\n        nn.init.normal_(self.embed.weight,std=0.05)\n        nn.init.normal_(self.pos.weight,std=0.01)\n    def forward(self,idx):\n        b,t=idx.shape\n        if t>self.context:raise ValueError(\'Input longer than context\')\n        h=self.embed(idx)+self.pos(torch.arange(t,device=idx.device))[None,:,:]\n        for block in self.blocks:h=block(h,self.shared_ff)\n        return F.linear(self.final_norm(h),self.embed.weight)\n    def set_pool_mode(self,mode):\n        for block in self.blocks:\n            if block.attn.pooled:block.attn.pool_mode=mode\n    def pool_stats(self):\n        pooled=[b.attn for b in self.blocks if b.attn.pooled]\n        if not pooled:return {}\n        return {\'gate\':float(np.mean([float(a.pool_logit.detach().sigmoid()) for a in pooled])),\n                \'pool_entropy_norm\':float(np.mean([a.last_entropy for a in pooled if a.last_entropy is not None]))\n                    if any(a.last_entropy is not None for a in pooled) else None}\n\n\nclass Mix(nn.Module):\n    """Same row/column mixing as earlier Matrix EXILRM; no flattened-state projection."""\n    def __init__(self,d,bias=False):\n        super().__init__(); self.a=nn.Linear(d,d,bias=bias); self.b=nn.Linear(d,d,bias=False)\n    def forward(self,x):return self.a(x)+self.b(x.transpose(-1,-2)).transpose(-1,-2)\n\n\ndef exponential_history(X, decay, backend=\'fft\'):\n    # X: B,T,H,D,D; decay: H. Strictly previous input, P[0]=0.\n    T=X.shape[1]; lam=decay.to(X.dtype); l=lam.view(1,-1,1,1)\n    if T==1:return X*0\n    if backend==\'scan\':\n        S=torch.zeros_like(X[:,0]); D=torch.zeros_like(l); rows=[]\n        for t in range(T):\n            rows.append(S/D.clamp_min(1)); S=X[:,t]+l*S; D=1+l*D\n        return torch.stack(rows,1)\n    if backend==\'parallel\':\n        # Inclusive affine prefix by doubling: A[t] = sum_{j<=t} lambda**(t-j)*X[j].\n        # Each stage uses only the previous stage; no in-place mutation or future reads.\n        # The denominator uses the identical prefix over ones. Shift to exclude X[t].\n        A=X; D=X.new_ones((1,T,X.shape[2],1,1)); stride=1\n        while stride<T:\n            factor=l[:,None].pow(stride)\n            A=torch.cat((A[:,:stride],A[:,stride:]+factor*A[:,:-stride]),dim=1)\n            D=torch.cat((D[:,:stride],D[:,stride:]+factor*D[:,:-stride]),dim=1)\n            stride*=2\n        return torch.cat((X[:,:1]*0,A[:,:-1]/D[:,:-1]),dim=1)\n    if backend!=\'fft\':raise ValueError(backend)\n    if T==1:return X*0\n    ex=torch.arange(T-1,device=X.device,dtype=X.dtype)[:,None]\n    weights=lam[None,:].pow(ex)\n    kernel=torch.cat([torch.zeros_like(weights[:1]),weights],0)\n    L=1<<(2*T-2).bit_length()\n    num=torch.fft.irfft(torch.fft.rfft(X,n=L,dim=1)*torch.fft.rfft(kernel,n=L,dim=0)[None,:,:,None,None],n=L,dim=1)[:,:T]\n    den=kernel.cumsum(0)[None,:,:,None,None]\n    return torch.cat([X[:,:1]*0,num[:,1:]/den[:,1:]],1)\n\n\ndef recurrent_chunk(M, a, b, c, q, ar, br, aw, bw, ac, bc, readnorm):\n    def mix(x,u,v):return F.linear(x,u)+F.linear(x.transpose(-1,-2),v).transpose(-1,-2)\n    ys=[]\n    for t in range(q.shape[1]):\n        r=torch.sigmoid(a[:,t]+mix(M,ar,br));w=torch.sigmoid(b[:,t]+mix(M,aw,bw))\n        C=torch.tanh(c[:,t]+mix(r*M,ac,bc));M=M+w*(C-M)\n        y=(M@q[:,t].unsqueeze(-1)).squeeze(-1)\n        y=y*torch.rsqrt(y.square().mean(-1,keepdim=True)+1e-5) if readnorm else y/math.sqrt(q.shape[-1])\n        ys.append(y)\n    return M,torch.stack(ys,1)\n\n_COMPILED_CHUNK=None\n\ndef compiled_chunk():\n    global _COMPILED_CHUNK\n    if _COMPILED_CHUNK is None:_COMPILED_CHUNK=torch.compile(recurrent_chunk,fullgraph=True,dynamic=False)\n    return _COMPILED_CHUNK\n\nclass MatrixLayer(nn.Module):\n    def __init__(self,width,heads,variant,backend):\n        super().__init__();self.h=heads;self.d=width//heads;self.variant=variant;self.backend=backend\n        self.kvq=nn.Linear(width,3*width);self.out=nn.Linear(width,width)\n        self.x=nn.ModuleList(Mix(self.d,True) for _ in range(3));self.p=nn.ModuleList(Mix(self.d) for _ in range(3));self.m=nn.ModuleList(Mix(self.d) for _ in range(3))\n        decay=torch.full((heads,),.9)\n        if variant==\'matrix_multi\':\n            decay=torch.tensor([.5,.8,.9,.95,.99,.995,.999,.9999]) if heads==8 else torch.exp(-torch.exp(torch.linspace(math.log(-math.log(.5)),math.log(-math.log(.9999)),heads)))\n        if variant in (\'matrix_learned\',\'matrix_multi\'):self.decay_logit=nn.Parameter(torch.logit(decay))\n        else:self.register_buffer(\'fixed_decay\',decay)\n        self.use_compile=False;self.readnorm=variant==\'matrix_readnorm\';self.capture_stats=False;self.last_gate=None\n    def decay(self):return self.decay_logit.sigmoid() if hasattr(self,\'decay_logit\') else self.fixed_decay\n    def forward(self,z):\n        B,T,W=z.shape;k,v,q=self.kvq(z).reshape(B,T,3,self.h,self.d).unbind(2)\n        # Keep FFT and recurrent memory in FP32 under AMP; preserve float64 in checks.\n        dtype=torch.float64 if z.dtype==torch.float64 else torch.float32\n        with torch.autocast(device_type=z.device.type,enabled=False):\n            k,v,q=(a.to(dtype) for a in (k,v,q));X=v.unsqueeze(-1)*k.unsqueeze(-2)\n            P=exponential_history(X,self.decay(),self.backend)\n            pre=[a(X)+b(P) for a,b in zip(self.x,self.p)]\n            M=X.new_zeros(B,self.h,self.d,self.d);reads=[]; gates=[]\n            if self.use_compile and z.is_cuda and self.training and not self.capture_stats:\n                params=[w for m in self.m for w in (m.a.weight,m.b.weight)]\n                for start in range(0,T,16):\n                    end=min(start+16,T);fn=compiled_chunk() if end-start==16 else recurrent_chunk\n                    M,y=fn(M,*(p[:,start:end] for p in pre),q[:,start:end],*params,self.readnorm)\n                    reads.extend(y.unbind(1))\n            else:\n                for t in range(T):\n                    r=torch.sigmoid(pre[0][:,t]+self.m[0](M));w=torch.sigmoid(pre[1][:,t]+self.m[1](M))\n                    C=torch.tanh(pre[2][:,t]+self.m[2](r*M)); M=M+w*(C-M)\n                    y=(M@q[:,t].unsqueeze(-1)).squeeze(-1)\n                    if self.readnorm:y=y*torch.rsqrt(y.square().mean(-1,keepdim=True)+1e-5)\n                    else:y=y/math.sqrt(self.d)\n                    reads.append(y)\n                    if self.capture_stats:gates.append(w.detach().mean())\n            if gates:self.last_gate=float(torch.stack(gates).mean().item())\n        return self.out(torch.stack(reads,1).reshape(B,T,W))\n\nclass MatrixBlock(nn.Module):\n    def __init__(self,cfg,hidden,variant):\n        super().__init__();d=cfg.width;self.norm_a=nn.LayerNorm(d);self.attn=MatrixLayer(d,cfg.matrix_heads,variant,cfg.history_backend);self.attn.use_compile=cfg.compile_chunks;self.norm_f=nn.LayerNorm(d);self.ff=MLP(d,hidden)\n    def forward(self,x,shared_ff=None):\n        z=x+self.attn(self.norm_a(x));return z+self.ff(self.norm_f(z))\n\n\ndef matrix_hidden(cfg,variant,vocab):\n    # Compute exact parameter counts analytically using real modules; only change FF width.\n    target=param_count(ReferenceModel(cfg,\'standard\',vocab))\n    block=MatrixBlock(cfg,cfg.hidden,"matrix_fixed")\n    base=vocab*cfg.width+cfg.layers*param_count(block)+2*cfg.width\n    h=max(8,round(cfg.hidden+(target-base)/(cfg.layers*(2*cfg.width+1))))\n    return h\n\nclass Model(nn.Module):\n    def __init__(self,cfg,variant,vocab_size=None):\n        super().__init__();self.variant=variant;self.context=cfg.context;self.width=cfg.width;v=vocab_size or cfg.vocab\n        if variant==\'standard\':\n            self.reference=ReferenceModel(cfg,\'standard\',v);self.ff_width=cfg.hidden\n        else:\n            self.embed=nn.Embedding(v,cfg.width);nn.init.normal_(self.embed.weight,std=.05)\n            self.ff_width=matrix_hidden(cfg,variant,v)\n            self.matrix_blocks=nn.ModuleList([MatrixBlock(cfg,self.ff_width,variant) for _ in range(cfg.layers)])\n            self.final_norm=nn.LayerNorm(cfg.width)\n    @property\n    def blocks(self):return self.reference.blocks if self.variant==\'standard\' else self.matrix_blocks\n    def forward(self,idx):\n        if self.variant==\'standard\':return self.reference(idx)\n        x=self.embed(idx)\n        for block in self.blocks:x=block(x)\n        return F.linear(self.final_norm(x),self.embed.weight)\n    def set_pool_mode(self,mode):\n        if mode!=\'normal\':raise ValueError(\'No pooling in this experiment\')\n    def pool_stats(self):\n        if self.variant==\'standard\':return {}\n        return {\'gate\':float(np.mean([b.attn.last_gate for b in self.blocks if b.attn.last_gate is not None])) if any(b.attn.last_gate is not None for b in self.blocks) else None}\n    def set_backend(self,backend):\n        if self.variant!=\'standard\':\n            for b in self.blocks:b.attn.backend=backend\n\n\ndef param_count(model):return sum(p.numel() for p in model.parameters())\n\n\ndef tensor_from_tokens(arr, offset, b, s, device):\n    n=b*s\n    x=np.asarray(arr[offset:offset+n],dtype=np.int64).reshape(b,s).copy()\n    y=np.asarray(arr[offset+1:offset+n+1],dtype=np.int64).reshape(b,s).copy()\n    return torch.as_tensor(x,device=device),torch.as_tensor(y,device=device)\n\n\ndef choose_batch(remain,cfg):\n    # Score exactly \'remain\' targets, even when a 1M boundary cuts a batch.\n    if remain>=cfg.context:\n        b=min(cfg.batch,remain//cfg.context)\n        return b,cfg.context\n    return 1,remain\n\n\ndef learning_rate(seen,cfg):\n    if seen<cfg.warmup_tokens:\n        return cfg.max_lr*max(0.03,seen/max(cfg.warmup_tokens,1))\n    t=min(1.0,(seen-cfg.warmup_tokens)/max(1,cfg.scored_tokens-cfg.warmup_tokens))\n    return cfg.min_lr+(cfg.max_lr-cfg.min_lr)*(1+math.cos(math.pi*t))*0.5\n\n\ndef amp_context(cfg,device):\n    if device.type!=\'cuda\' or not cfg.use_amp:return nullcontext()\n    return torch.autocast(device_type=\'cuda\',dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16))\n\n\ndef make_scaler(cfg,device):\n    enabled=device.type==\'cuda\' and cfg.use_amp and not torch.cuda.is_bf16_supported()\n    try:return torch.amp.GradScaler(\'cuda\',enabled=enabled)\n    except TypeError:return torch.cuda.amp.GradScaler(enabled=enabled)\n\n\ndef model_grad_norm(model,keyword):\n    sums=0.\n    for name,p in model.named_parameters():\n        if keyword in name and p.grad is not None:sums+=float(p.grad.detach().float().square().sum())\n    return sums**0.5\n\n\ndef model_checkpoint_paths(cfg,name):\n    d=Path(cfg.root)/\'checkpoints\'/name\n    d.mkdir(parents=True,exist_ok=True)\n    return d,d/\'latest.pt\'\n\n\ndef atomic_torch(path,obj):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.part\')\n    torch.save(obj,tmp); tmp.replace(path)\n\n\ndef export_checkpoint(cfg,name,model,optim,scaler,seen,train_steps,manifest_hash,reason=\'milestone\'):\n    folder,latest=model_checkpoint_paths(cfg,name)\n    state={\'version\':VERSION,\'variant\':name,\'token_cursor\':seen,\'train_steps\':train_steps,\n           \'manifest_hash\':manifest_hash,\'cfg\':asdict(cfg),\'model\':model.state_dict(),\n           \'optimizer\':optim.state_dict(),\'scaler\':scaler.state_dict(),\n           \'reason\':reason,\n           \'cpu_rng\':torch.get_rng_state(),\n           \'cuda_rng\':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}\n    atomic_torch(latest,state)\n    if seen and (seen%cfg.archive_every==0 or seen==cfg.scored_tokens):\n        snap=folder/(f\'{seen//1_000_000:04d}M.pt\')\n        if not snap.exists():shutil.copy2(latest,snap)\n    atomic_json(folder/\'progress.json\',{\'variant\':name,\'token_cursor\':seen,\'train_steps\':train_steps,\'reason\':reason})\n\n\ndef try_restore(cfg,name,model,optimizer,scaler,manifest_hash):\n    folder,latest=model_checkpoint_paths(cfg,name)\n    if not latest.exists():return 0,0\n    try:state=torch.load(latest,map_location=\'cpu\',weights_only=False)\n    except TypeError:state=torch.load(latest,map_location=\'cpu\')\n    if state[\'variant\']!=name or state[\'manifest_hash\']!=manifest_hash or state[\'version\']!=VERSION:\n        raise RuntimeError(\'Refusing checkpoint whose dataset/model experiment identity differs\')\n    model.load_state_dict(state[\'model\']);optimizer.load_state_dict(state[\'optimizer\'])\n    scaler.load_state_dict(state[\'scaler\'])\n    # No dropout; per-model stochastic state is not needed in main training path.\n    print(\'RESUMED\',name,\'at\',f"{state[\'token_cursor\']:,}",\'tokens\',flush=True)\n    return int(state[\'token_cursor\']),int(state[\'train_steps\'])\n\n\ndef copy_previous_session(cfg):\n    """Optional Kaggle Input source for previous /kaggle/working output version."""\n    if not cfg.resume_input_root:return\n    src=Path(cfg.resume_input_root)\n    if not src.exists():raise FileNotFoundError(\'resume_input_root does not exist: \'+str(src))\n    out=Path(cfg.root);out.mkdir(parents=True,exist_ok=True)\n    # Locate exact experiment root or a parent with a direct child.\n    if not (src/\'experiment_config.json\').exists():\n        found=list(src.glob(\'**/experiment_config.json\'))\n        if len(found)!=1:raise RuntimeError(\'Supply resume_input_root containing one experiment_config.json; found \'+str(len(found)))\n        src=found[0].parent\n    print(\'Restoring previous Kaggle session from\',src,flush=True)\n    for folder in (\'data\',\'checkpoints\',\'metrics\',\'probes\',\'reports\'):\n        origin=src/folder\n        if not origin.exists():continue\n        dest=out/folder;dest.mkdir(exist_ok=True)\n        for f in origin.rglob(\'*\'):\n            if not f.is_file():continue\n            target=dest/f.relative_to(origin)\n            if target.exists():continue\n            target.parent.mkdir(parents=True,exist_ok=True)\n            shutil.copy2(f,target)\n    for fn in (\'experiment_config.json\',\'experiment_manifest.json\',\'training_log.txt\',\'model_specs.json\'):\n        f=src/fn\n        if f.exists() and not (out/fn).exists():shutil.copy2(f,out/fn)\n\n\ndef sanity_gradients(cfg,device,vocab_size):\n    rows=[]\n    for name in cfg.models:\n        seed_everything(cfg.seed+17);m=Model(cfg,name,vocab_size).to(device)\n        x=torch.randint(0,vocab_size,(2,16),device=device);logits=m(x)\n        loss=F.cross_entropy(logits[:,:-1].reshape(-1,vocab_size),x[:,1:].reshape(-1));loss.backward()\n        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in m.parameters()),name\n        with torch.no_grad():\n            y=x.clone();y[:,8:]=(y[:,8:]+1)%vocab_size\n            err=(m(x)[:,:8]-m(y)[:,:8]).abs().max().item();assert err<1e-4,(name,err)\n        row={\'model\':name,\'loss\':float(loss.detach()),\'causality_max_error\':err,\'params\':param_count(m)}\n        if name!=\'standard\':\n            m.zero_grad(set_to_none=True);m.set_backend(\'fft\');a=m(x);a.square().mean().backward();ga=[p.grad.clone() for p in m.parameters()]\n            m.zero_grad(set_to_none=True);m.set_backend(\'scan\');b=m(x);b.square().mean().backward()\n            row[\'fft_scan_logits_max_error\']=(a-b).abs().max().item();row[\'fft_scan_grad_max_error\']=max((p.grad-g).abs().max().item() for p,g in zip(m.parameters(),ga))\n            assert row[\'fft_scan_logits_max_error\']<1e-4 and row[\'fft_scan_grad_max_error\']<1e-4,row\n        rows.append(row);del m\n    # Independent direct float64 history check including learnable decay gradients.\n    X=torch.randn(2,13,3,4,4,dtype=torch.float64,device=device,requires_grad=True);lam=torch.tensor([.5,.9,.9999],dtype=torch.float64,device=device,requires_grad=True)\n    direct=[X[:,0]*0]\n    for t in range(1,13):\n        w=lam[None,:].pow(torch.arange(t-1,-1,-1,device=device,dtype=X.dtype)[:,None]);direct.append((X[:,:t]*w[None,:,:,None,None]).sum(1)/w.sum(0)[None,:,None,None])\n    direct=torch.stack(direct,1);a=exponential_history(X,lam,\'fft\');b=exponential_history(X,lam,\'scan\')\n    assert (a-direct).abs().max()<1e-11 and (b-direct).abs().max()<1e-11\n    ga=torch.autograd.grad(a.square().sum(),(X,lam));gb=torch.autograd.grad(b.square().sum(),(X,lam))\n    assert max((u-v).abs().max().item() for u,v in zip(ga,gb))<1e-9\n    # Check the optional short-unroll implementation against the literal recurrence.\n    m=MatrixLayer(32,8,\'matrix_fixed\',\'fft\').to(device).eval();z=torch.randn(2,16,32,device=device)\n    with torch.no_grad():\n        k,v,q=m.kvq(z).reshape(2,16,3,8,4).unbind(2);X=v.unsqueeze(-1)*k.unsqueeze(-2)\n        P=exponential_history(X,m.decay(),\'fft\');pre=[a(X)+b(P) for a,b in zip(m.x,m.p)]\n        M,y=recurrent_chunk(X.new_zeros(2,8,4,4),*pre,q,*(w for a in m.m for w in (a.a.weight,a.b.weight)),False)\n        assert (m.out(y.reshape(2,16,32))-m(z)).abs().max().item()<1e-5\n    seed_everything(cfg.seed);return rows\n\n\ndef window_audit(model,arr,cfg,device,name,seen,split):\n    """Fixed targets; full recurrent prefix vs recent-only control; no training."""\n    rng=np.random.default_rng(cfg.seed+(12345 if split==\'test\' else 23456)); maxw=max(cfg.context,2048)\n    if len(arr)<=maxw+2:return\n    targets=rng.integers(maxw,len(arr),size=cfg.window_examples); replacements=rng.integers(0,len(arr)-maxw,size=cfg.window_examples);rows=[]\n    model.eval()\n    with torch.inference_mode(),amp_context(cfg,device):\n        for W in sorted(set([32,128,cfg.context,1024,2048])):\n            for i,t in enumerate(targets):\n                t=int(t);eff=min(W,cfg.context) if name==\'standard\' else W\n                x=torch.tensor(np.array(arr[t-eff:t],dtype=np.int64),device=device)[None];truth=int(arr[t])\n                a=model(x)[0,-1].float();recent=min(64,eff);b=model(x[:,-recent:])[0,-1].float()\n                # unrelated prefix, identical last recent tokens\n                if eff>recent:\n                    j=int(replacements[i]);bad=x.clone();bad[:,:eff-recent]=torch.tensor(np.array(arr[j:j+eff-recent],dtype=np.int64),device=device)\n                    c=model(bad)[0,-1].float();delta=float((c-a).abs().mean());gain=float(F.log_softmax(a,0)[truth]-F.log_softmax(c,0)[truth])\n                else:delta=None;gain=None\n                rows.append({\'model\':name,\'tokens\':seen,\'split\':split,\'example\':i,\'requested_context\':W,\'actual_context\':eff,\'target_offset\':t,\'nll\':float(-F.log_softmax(a,0)[truth]),\'recent_nll\':float(-F.log_softmax(b,0)[truth]),\'old_prefix_logit_change\':delta,\'old_prefix_nll_gain\':gain})\n    for r in rows:append_csv(Path(cfg.root)/\'metrics\'/\'window_audit.csv\',r)\n\n\n\ndef real_repeat_audit(model,arr,cfg,device,name,seen,split):\n    rng=np.random.default_rng(cfg.seed+(551 if split==\'test\' else 552));model.eval()\n    if len(arr)<4096:return\n    rows=[]\n    with torch.inference_mode(),amp_context(cfg,device):\n        for gap in (0,32,128,384,1024,2048):\n            for i in range(cfg.window_examples):\n                a,b,c=(int(v) for v in rng.integers(0,len(arr)-4096,size=3));p=list(map(int,arr[a:a+17]));wrong=list(map(int,arr[b:b+17]));f=list(map(int,arr[c:c+gap]));truth=p[-1]\n                good=p+f+p[:-1];bad=wrong+f+p[:-1];full=len(good)\n                if name==\'standard\':good=good[-cfg.context:];bad=bad[-cfg.context:]\n                def nll(ids):\n                    x=torch.tensor(ids,device=device)[None];return float(-F.log_softmax(model(x)[0,-1].float(),0)[truth])\n                g=nll(good);d=nll(bad)\n                rows.append({\'model\':name,\'tokens\':seen,\'split\':split,\'gap\':gap,\'example\':i,\'requested_context\':full,\'actual_context\':len(good),\'nll_repeat\':g,\'nll_control\':d,\'benefit_nats\':d-g})\n    for row in rows:append_csv(Path(cfg.root)/\'metrics\'/\'real_repeat.csv\',row)\n\ndef long_fact_audit(model,sp,arr,cfg,device,name,seen,split):\n    """Ordinary text only; paired full-prefix/recent-window evaluation on held-out facts.\n    Filler comes from held-out real text. Each model sees the same facts and questions.\n    The same model\'s suffix condition distinguishes access to old facts from architecture.\n    """\n    vals=eligible_values(sp)\n    if len(vals)<4 or len(arr)<4096:return\n    rng=random.Random(cfg.seed+(981231 if split==\'test\' else 981230));model.eval();rows=[]\n    for gap in (0,128,384,512,1024,2048):\n        for i in range(cfg.window_examples):\n            keys=rng.sample(NAMES,4);chosen=rng.sample(vals,4);j=rng.randrange(4);truth=chosen[j];options=chosen.copy();rng.shuffle(options)\n            order=list(range(4));rng.shuffle(order)\n            facts=\'\'.join(f\'{keys[k]} = {chosen[k][0]}\\n\' for k in order)\n            query=f\'\\nQuestion: What value belongs to {keys[j]}?\\nAnswer:\'\n            a=rng.randrange(0,len(arr)-max(gap,1));filler=list(map(int,arr[a:a+gap]));ids=sp.encode(facts,out_type=int)+filler+sp.encode(query,out_type=int)\n            # Never use a learned-position Transformer beyond its configured table.\n            for mode in (\'full\',\'recent\'):\n                if name==\'standard\' and mode==\'full\' and len(ids)>cfg.context:continue\n                actual=ids[-cfg.context:] if mode==\'recent\' else ids\n                pr={\'valid\':True,\'tokens\':actual,\'truth\':truth,\'options\':options}\n                pred=score_one_probe(model,pr,device,cfg)\n                rows.append({\'model\':name,\'tokens\':seen,\'split\':split,\'gap\':gap,\'example\':i,\'mode\':mode,\'requested_context\':len(ids),\'actual_context\':len(actual),\'choice_correct\':pred[\'choice_correct\'],\'unrestricted_correct\':pred[\'unrestricted_correct\'],\'truth_id\':truth[1],\'prediction_id\':pred[\'pred_id\']})\n    for row in rows:append_csv(Path(cfg.root)/\'metrics\'/\'long_fact_audit.csv\',row)\n\n\ndef backend_audit(model,cfg,device,name,seen):\n    if name==\'standard\':return\n    model.eval();x=torch.arange(min(cfg.context,128),device=device)[None]%model.embed.num_embeddings\n    times={};outputs={}\n    with torch.inference_mode():\n        for backend in (\'fft\',\'scan\',\'parallel\'):\n            model.set_backend(backend);model(x)\n            if device.type==\'cuda\':torch.cuda.synchronize()\n            start=time.perf_counter()\n            for _ in range(3):outputs[backend]=model(x).float()\n            if device.type==\'cuda\':torch.cuda.synchronize()\n            times[backend]=(time.perf_counter()-start)/3\n    model.set_backend(cfg.history_backend)\n    append_csv(Path(cfg.root)/\'metrics\'/\'backend_audit.csv\',{\'model\':name,\'tokens\':seen,\'length\':x.shape[1],\'fft_seconds\':times[\'fft\'],\'scan_seconds\':times[\'scan\'],\'parallel_seconds\':times[\'parallel\'],\'max_logit_error\':float((outputs[\'fft\']-outputs[\'scan\']).abs().max()),\'parallel_max_logit_error\':float((outputs[\'fft\']-outputs[\'parallel\']).abs().max())})\n\n\ndef evaluate_nll(model,arr,cfg,device,max_positions=8192,offset=0):\n    model.eval();total=0.;correct=0;count=0;distinct=set();windows=[]\n    length=min(len(arr)-offset-1,max_positions)\n    with torch.inference_mode():\n        pos=offset\n        while pos < offset+length:\n            remain=offset+length-pos\n            b=min(cfg.batch,max(1,remain//cfg.context))\n            s=cfg.context if remain>=cfg.context else remain\n            n=min(b*s,remain)\n            if n!=b*s:b=1;s=n\n            x,y=tensor_from_tokens(arr,pos,b,s,device)\n            with amp_context(cfg,device):logits=model(x)\n            scores=logits.float()\n            loss=F.cross_entropy(scores.reshape(-1,scores.size(-1)),y.reshape(-1),reduction=\'sum\')\n            total+=float(loss.item());count+=n\n            p=scores.argmax(-1)\n            correct+=int((p==y).sum().item())\n            distinct.update(p.reshape(-1).unique().detach().cpu().tolist())\n            windows.append({\'tokens\':n,\'nll\':float(loss.item())/n})\n            pos+=n\n    return {\'nll\':total/max(count,1),\'ppl\':math.exp(min(total/max(count,1),30)),\n            \'token_accuracy\':correct/max(count,1),\'unique_predicted_tokens\':len(distinct),\n            \'positions\':count,\'window_nll\':windows}\n\n# ------------------ Held-out synthetic probes ------------------\nNAMES=\'Alice Bob Carol David Emma Frank Grace Henry Iris Jack Kate Liam Mia Noah Olivia Paul Quinn Rose Sam Tara Uma Vera Will Xena Yara Zach Abel Bella Chloe Damon Elsa Felix Gina Hugo Ivan Jane Kira Leo Maya Nina Omar Pia Rex Sara Theo Una Veda Wade Yvonne Zoe Aaron Basil Clara Dina Eric Faye Garth Hope Isaac Juno Kian Lora Mira Nora Opal Peter Rina Seth Tina Vick Wendy Xander\'.split()\nVALUES=\'red blue green gold white black brown silver pink orange purple yellow copper violet amber coral indigo pearl ivory teal gray grey cream bronze lime rose violet scarlet olive ruby navy ochre jade iron stone water fire earth wind north south east west alpha beta gamma delta cedar maple oak pine birch moss cloud rain snow sun moon star dawn dusk light dark calm swift brave quiet bright warm cold small large soft sharp clean round true false near far high low first last\'.split()\n\n\nclass InsufficientProbeVocabulary(ValueError):\n    """An individual synthetic probe is untestable with the current tokenizer.\n\n    This must become a recorded SKIP_VOCAB, not a training-stopping exception.\n    """\n    def __init__(self, required, available):\n        self.required=int(required); self.available=int(available)\n        super().__init__(f\'Probe needs {required} distinct answer tokens, tokenizer offers {available}\')\n\n\n@functools.lru_cache(maxsize=8)\ndef eligible_values(sp):\n    """All eligible *single-token* answer words, not just the original short list.\n\n    Important fix: the 32-fact probe requires >=32 distinct answers. The old\n    implementation expanded the vocabulary only for <8; it crashed at 10M.\n    Here we always expand from clean SentencePiece whole-word pieces and verify\n    that prepending a space actually encodes as exactly that one token.\n    """\n    vals=[]; ids=set(); words=set()\n    def add(word):\n        word=word.strip().lower()\n        if len(word)<3 or len(word)>24 or not word.isascii() or not word.isalpha():return\n        if word in words or word.capitalize() in NAMES:return\n        ids1=sp.encode(\' \'+word,out_type=int)\n        if len(ids1)==1 and ids1[0]!=sp.unk_id() and ids1[0] not in ids:\n            vals.append((word,int(ids1[0]))); ids.add(int(ids1[0])); words.add(word)\n    for word in VALUES:add(word)\n    # Don\'t wait until fewer than 8 words: 32-fact tests need 32 distinct values.\n    for i in range(sp.vocab_size()):\n        piece=sp.id_to_piece(i)\n        if piece.startswith(\'▁\') and len(piece)>3:\n            add(piece[1:])\n    return tuple(vals)\n\n\n@functools.lru_cache(maxsize=8)\ndef filler_pieces(sp):\n    pieces=[]\n    for w in (\' the\',\' and\',\' but\',\' before\',\' after\',\' when\',\' with\',\' other\',\' only\',\' near\',\' the\'):\n        z=sp.encode(w,out_type=int)\n        if len(z)==1 and z[0]!=sp.unk_id():pieces.append(int(z[0]))\n    if not pieces:\n        vals=eligible_values(sp)\n        if vals:pieces=[vals[0][1]]\n        else:\n            # For a pathological tokenizer, select any valid ordinary token.\n            for i in range(sp.vocab_size()):\n                if i not in {sp.unk_id(),sp.eos_id(),sp.bos_id(),sp.pad_id()}:\n                    pieces=[i];break\n    return tuple(pieces)\n\n\ndef _compose_probe_core(sp,kind,condition,index,seed,context):\n    rng=random.Random(seed*10000019+index*11087+sum(ord(c) for c in kind+str(condition))*983)\n    vals=eligible_values(sp)\n    names=rng.sample(NAMES,min(len(NAMES),44))\n    def pick_values(k):\n        if len(vals)<k:raise InsufficientProbeVocabulary(k,len(vals))\n        return rng.sample(vals,k)\n\n    def select_options(truth,include=()):\n        if len(vals)<4:raise InsufficientProbeVocabulary(4,len(vals))\n        picked=[truth]\n        for p in include:\n            if p[1] not in [x[1] for x in picked] and len(picked)<4:picked.append(p)\n        for p in rng.sample(vals,len(vals)):\n            if len(picked)>=4:break\n            if p[1] not in [x[1] for x in picked]:picked.append(p)\n        rng.shuffle(picked)\n        return picked\n    def qtokens(text):return sp.encode(text,out_type=int)\n    fill=filler_pieces(sp)\n    def build(facts,question,gap=0):\n        gap_tokens=[fill[i%len(fill)] for i in range(gap)]\n        return qtokens(facts)+gap_tokens+qtokens(question)\n    pid=f\'{kind}:{condition}:{index}:{seed}\'\n    common={\'probe_id\':pid,\'category\':kind,\'condition\':str(condition),\'seed\':seed,\'index\':index}\n    out=[]\n    if kind==\'binding\':\n        n=int(condition)\n        chosen=pick_values(n)\n        mapping=list(zip(names[:n],chosen))\n        rng.shuffle(mapping)\n        key,truth=rng.choice(mapping)\n        facts=\'\'.join(f\'{k} = {v[0]}\\n\' for k,v in mapping)\n        ids=build(facts,f\'\\nQuestion: What value belongs to {key}?\\nAnswer:\')\n        out=[dict(common,tokens=ids,truth=truth,options=select_options(truth,[v for k,v in mapping if v!=truth]),role=\'target\')]\n    elif kind==\'recall\':\n        gap=int(condition);truth=pick_values(1)[0];key=names[0]\n        ids=build(f\'{key} = {truth[0]}\\n\',f\'\\nQuestion: What value belongs to {key}?\\nAnswer:\',gap)\n        out=[dict(common,tokens=ids,truth=truth,options=select_options(truth),role=\'target\',gap=gap)]\n    elif kind==\'distance\':\n        gap=int(condition)\n        chosen=pick_values(4);mapping=list(zip(names[:4],chosen));key,truth=rng.choice(mapping);rng.shuffle(mapping)\n        facts=\'\'.join(f\'{k} = {v[0]}\\n\' for k,v in mapping)\n        ids=build(facts,f\'\\nQuestion: What value belongs to {key}?\\nAnswer:\',gap)\n        out=[dict(common,tokens=ids,truth=truth,options=select_options(truth,chosen),role=\'target\',gap=gap)]\n    elif kind==\'multi_hop\':\n        hops=int(condition);chosen=pick_values(4);chains=[names[i*hops:(i+1)*hops] for i in range(4)];rows=[]\n        for chain,val in zip(chains,chosen):\n            rows.extend(f\'{chain[i]} leads to {chain[i+1]}\\n\' for i in range(hops-1));rows.append(f\'{chain[-1]} leads to {val[0]}\\n\')\n        rng.shuffle(rows);j=rng.randrange(4);truth=chosen[j]\n        ids=build(\'\'.join(rows),f\'\\nQuestion: Starting from {chains[j][0]}, what final value is reached?\\nAnswer:\')\n        out=[dict(common,tokens=ids,truth=truth,options=select_options(truth,chosen),role=\'target\',hops=hops)]\n    elif kind==\'overwrite\':\n        vals4=pick_values(4)\n        old,new,other,extra=vals4[:4]\n        alice,bob,carol=names[:3]\n        facts=f\'{alice} = {old[0]}\\n{bob} = {other[0]}\\n{carol} = {extra[0]}\\nCorrection: {alice} = {new[0]}\\n\'\n        for role,key,truth in [(\'updated\',alice,new),(\'unchanged\',bob,other)]:\n            ids=build(facts,f\'\\nQuestion: What is the current value of {key}?\\nAnswer:\')\n            out.append(dict(common,probe_id=pid+\':\'+role,tokens=ids,truth=truth,\n                       options=select_options(truth,[old,new,other,extra]),role=role))\n    elif kind==\'counterfactual\':\n        old,new,other,extra=pick_values(4)\n        alice,bob=names[:2]\n        stem=f\'{alice} = {old[0]}\\n{bob} = {other[0]}\\n\'\n        changed=f\'{alice} = {new[0]}\\n{bob} = {other[0]}\\n\'\n        opts=select_options(old,[new,other,extra])\n        settings=[(\'target_before\',stem,alice,old),(\'target_after\',changed,alice,new),\n                  (\'unrelated_before\',stem,bob,other),(\'unrelated_after\',changed,bob,other)]\n        for role,txt,key,truth in settings:\n            ids=build(txt,f\'\\nQuestion: What is the value for {key}?\\nAnswer:\')\n            out.append(dict(common,probe_id=pid+\':\'+role,pair_id=pid,\n                            tokens=ids,truth=truth,options=opts,role=role))\n    elif kind==\'query_swap\':\n        val1,val2,extra1,extra2=pick_values(4)\n        alice,bob=names[:2]\n        facts=f\'{alice} = {val1[0]}\\n{bob} = {val2[0]}\\n\'\n        opts=select_options(val1,[val2,extra1,extra2])\n        for role,key,truth in [(\'first\',alice,val1),(\'second\',bob,val2)]:\n            ids=build(facts,f\'\\nQuestion: What is the value for {key}?\\nAnswer:\')\n            out.append(dict(common,probe_id=pid+\':\'+role,pair_id=pid,\n                     tokens=ids,truth=truth,options=opts,role=role))\n    else:raise ValueError(kind)\n    for item in out:\n        item[\'valid\']=(len(item[\'tokens\'])>=2 and len(item[\'tokens\'])<=context and\n                       len({v[1] for v in item[\'options\']})==4)\n        item[\'invalid_reason\']=\'\' if item[\'valid\'] else (\'SKIP_CONTEXT\' if len(item[\'tokens\'])>context else \'SKIP_CANDIDATES\')\n        item[\'prompt_hash\']=hashlib.sha256(np.asarray(item[\'tokens\'],dtype=np.uint16).tobytes()).hexdigest()[:20]\n    return out\n\n\ndef compose_probe(sp,kind,condition,index,seed,context):\n    """Non-throwing probe construction: do not waste a 500M training run on a bad probe.\n\n    If a tokenizer cannot support a condition, keep an explicit skipped record\n    with its category and expected role; never replace missing answers by\n    duplicates (which would corrupt four-choice accuracy).\n    """\n    try:\n        return _compose_probe_core(sp,kind,condition,index,seed,context)\n    except InsufficientProbeVocabulary as e:\n        pid=f\'{kind}:{condition}:{index}:{seed}\'\n        role_sets={\'overwrite\':(\'updated\',\'unchanged\'),\n                   \'counterfactual\':(\'target_before\',\'target_after\',\'unrelated_before\',\'unrelated_after\'),\n                   \'query_swap\':(\'first\',\'second\')}\n        roles=role_sets.get(kind,(\'target\',))\n        return [{\'probe_id\':pid+(\':\'+role if len(roles)>1 else \'\'),\n                 \'pair_id\':pid if len(roles)>1 else \'\',\n                 \'category\':kind,\'condition\':str(condition),\'seed\':seed,\n                 \'index\':index,\'role\':role,\'tokens\':[], \'truth\':(\'\',-1),\n                 \'options\':[], \'valid\':False,\'prompt_hash\':\'\',\n                 \'invalid_reason\':\'SKIP_VOCAB\', \'needed_values\':e.required,\n                 \'available_values\':e.available}\n                for role in roles]\n\n\ndef probe_specs(stage, cfg, test=False):\n    if stage==\'mini\':\n        return [(\'binding\',2,4),(\'binding\',8,4),(\'overwrite\',\'facts\',4),(\'multi_hop\',2,4)]\n    if stage==\'regular\':\n        return [(\'binding\',n,8) for n in [2,4,8,16,32]] + \\\n               [(\'distance\',g,8) for g in [0,16,64,128,256,384]]+[(\'recall\',g,8) for g in [0,16,64,128,256,384]] + \\\n               [(\'multi_hop\',n,8) for n in [2,3,4]] + \\\n               [(\'overwrite\',\'facts\',8),(\'counterfactual\',\'edits\',8),(\'query_swap\',\'swap\',8)]\n    if stage==\'full\':\n        per=64 if test else 16\n        return [(\'binding\',n,per) for n in [2,4,8,16,32]]+ \\\n               [(\'distance\',g,per) for g in [0,16,64,128,256,384]]+[(\'recall\',g,per) for g in [0,16,64,128,256,384]]+ \\\n               [(\'multi_hop\',n,per) for n in [2,3,4]]+ \\\n               [(\'overwrite\',\'facts\',per),(\'counterfactual\',\'edits\',per),(\'query_swap\',\'swap\',per)]\n    raise ValueError(stage)\n\n\ndef probe_preflight(sp,cfg):\n    """Exercise every probe condition before GPU training; record coverage.\n\n    This deliberately validates, rather than deleting, difficult conditions.\n    Conditions without enough clean single-token answers are marked SKIP_VOCAB.\n    It also checks exact four-way answer distinctness, paired query structure,\n    tokenizer token ID bounds, and context-length handling.\n    """\n    report={\'vocab_size\':sp.vocab_size(), \'eligible_answer_values\':len(eligible_values(sp)),\n            \'context\':cfg.context,\'conditions\':[], \'invalid_reasons\':{},\n            \'checked_probes\':0,\'valid_probes\':0,\'skipped_probes\':0}\n    for stage in (\'mini\',\'regular\',\'full\'):\n        for kind,condition,_ in probe_specs(stage,cfg,test=False):\n            # Several seeds/indices to catch longer tokenised names/values.\n            for i in (0,1,7,31,63):\n                probes=compose_probe(sp,kind,condition,i,cfg.seed+700_009,cfg.context)\n                if not probes:raise AssertionError(\'Probe generator returned no records\')\n                for pr in probes:\n                    report[\'checked_probes\']+=1\n                    if pr[\'valid\']:\n                        report[\'valid_probes\']+=1\n                        assert 2 <=len(pr[\'tokens\'])<=cfg.context\n                        assert len(pr[\'options\'])==4\n                        assert len(set(v[1] for v in pr[\'options\']))==4\n                        assert pr[\'truth\'][1] in [v[1] for v in pr[\'options\']]\n                        assert all(0<=id_<sp.vocab_size() for id_ in pr[\'tokens\'])\n                    else:\n                        report[\'skipped_probes\']+=1\n                        reason=pr.get(\'invalid_reason\') or \'INVALID\'\n                        report[\'invalid_reasons\'][reason]=report[\'invalid_reasons\'].get(reason,0)+1\n                    report[\'conditions\'].append({\'stage\':stage,\'kind\':kind,\'condition\':str(condition),\n                         \'index\':i,\'role\':pr[\'role\'],\'status\':\'OK\' if pr[\'valid\'] else pr[\'invalid_reason\']})\n    atomic_json(Path(cfg.root)/\'reports\'/\'probe_preflight.json\',report)\n    print(\'PROBE PREFLIGHT:\',report[\'valid_probes\'],\'valid,\',\n          report[\'skipped_probes\'],\'skipped; reasons=\',report[\'invalid_reasons\'],flush=True)\n    return report\n\n\ndef score_one_probe(model,pr,device,cfg):\n    # Stable output schema for *both* successful and skipped probes; CSV writers\n    # must not silently move columns when the first condition is skipped.\n    out={\'status\':\'OK\',\'choice_correct\':None,\'unrestricted_correct\':None,\n         \'prediction\':None,\'pred_id\':None,\'truth\':pr[\'truth\'][0] or None,\n         \'truth_id\':pr[\'truth\'][1] if pr[\'truth\'][1]>=0 else None,\n         \'margin\':None,\'confidence\':None,\n         \'skip_reason\':pr.get(\'invalid_reason\',\'\'),\n         \'needed_values\':pr.get(\'needed_values\'),\n         \'available_values\':pr.get(\'available_values\')}\n    if not pr[\'valid\']:\n        out[\'status\']=pr.get(\'invalid_reason\') or \'SKIP_CONTEXT\'\n        return out\n    ctx=torch.as_tensor(pr[\'tokens\'],dtype=torch.long,device=device).unsqueeze(0)\n    with torch.inference_mode(),amp_context(cfg,device):logits=model(ctx)\n    last=logits[0,-1,:].float()\n    choices=[v[1] for v in pr[\'options\']]\n    scores=last[choices]\n    win=int(scores.argmax().item())\n    probs=scores.softmax(-1)\n    out.update({\'choice_correct\':int(choices[win]==pr[\'truth\'][1]),\n            \'unrestricted_correct\':int(last.argmax().item()==pr[\'truth\'][1]),\n            \'prediction\':pr[\'options\'][win][0], \'pred_id\':choices[win],\n            \'margin\':float((scores[win]-scores.topk(2).values[1]).item()),\n            \'confidence\':float(probs[win].item())})\n    return out\n\n\ndef run_probes(model,sp,cfg,device,name,milestone,stage=\'regular\',test=False,pool_mode=\'normal\',max_cases=None):\n    model.eval();model.set_pool_mode(pool_mode)\n    out=Path(cfg.root)\n    folder=out/\'probes\' / (\'test\' if test else \'validation\')\n    folder.mkdir(parents=True,exist_ok=True)\n    csvname=\'probe_rows.csv\' if not test else \'final_probe_rows.csv\'\n    rows=[];skips=defaultdict(int)\n    # Deterministic frozen probes, separate validation/test IDs.\n    probe_seed=cfg.seed+(99_000_009 if test else 700_009)\n    k=0\n    for kind,condition,n in probe_specs(stage,cfg,test):\n        for j in range(n):\n            for p in compose_probe(sp,kind,condition,j,probe_seed,cfg.context):\n                if max_cases is not None and k>=max_cases:break\n                k+=1\n                prediction=score_one_probe(model,p,device,cfg)\n                row={\'model\':name,\'token_milestone\':milestone,\'split\':\'test\' if test else \'validation\',\n                     \'pool_mode\':pool_mode,\'stage\':stage,\'probe_id\':p[\'probe_id\'],\n                     \'pair_id\':p.get(\'pair_id\',\'\'),\'kind\':p[\'category\'],\n                     \'condition\':p[\'condition\'],\'role\':p[\'role\'],\'prompt_sha256_20\':p[\'prompt_hash\'],\n                     \'context_tokens\':len(p[\'tokens\']), \'candidate_token_ids\':\';\'.join(str(x[1]) for x in p[\'options\']),\n                     \'correct_token_id\':p[\'truth\'][1],**prediction}\n                rows.append(row)\n                if prediction[\'status\']!=\'OK\':skips[kind]+=1\n            if max_cases is not None and k>=max_cases:break\n        if max_cases is not None and k>=max_cases:break\n    if not rows:return {\'count\':0,\'accuracy\':None,\'skips\':dict(skips)}\n    for row in rows:append_csv(folder/csvname,row)\n    valid=[r for r in rows if r[\'choice_correct\'] is not None]\n    categories={}\n    for kind in sorted({r[\'kind\'] for r in rows}):\n        rv=[r for r in valid if r[\'kind\']==kind]\n        categories[kind]={\'n\':len(rv),\'accuracy\':sum(r[\'choice_correct\'] for r in rv)/len(rv) if rv else None,\n                          \'unrestricted_accuracy\':sum(r[\'unrestricted_correct\'] for r in rv)/len(rv) if rv else None}\n    pairs=defaultdict(list)\n    for r in valid:\n        if r[\'pair_id\']:pairs[r[\'pair_id\']].append(r[\'choice_correct\'])\n    complete_pairs=[v for k,v in pairs.items() if len(v)==(4 if k.startswith(\'counterfactual:\') else 2)]\n    summary={\'pair_all_correct\':float(np.mean([all(v) for v in complete_pairs])) if complete_pairs else None,\'complete_pairs\':len(complete_pairs),\'model\':name,\'tokens\':milestone,\'stage\':stage,\'split\':\'test\' if test else \'validation\',\n         \'pool_mode\':pool_mode,\'count\':len(valid),\'skipped\':len(rows)-len(valid),\n         \'accuracy\':sum(r[\'choice_correct\'] for r in valid)/len(valid) if valid else None,\n         \'unrestricted_accuracy\':sum(r[\'unrestricted_correct\'] for r in valid)/len(valid) if valid else None,\n         \'categories\':categories}\n    with (folder/\'probe_summaries.jsonl\').open(\'a\') as f:f.write(json.dumps(summary)+\'\\n\')\n    model.set_pool_mode(\'normal\')\n    return summary\n\n\ndef score_suffix(model,prefix,suffix,device,cfg):\n    ids=prefix+suffix\n    if len(prefix)<1 or len(ids)-1>cfg.context:return None\n    x=torch.tensor(ids[:-1],device=device,dtype=torch.long).unsqueeze(0)\n    with torch.inference_mode(),amp_context(cfg,device):logits=model(x)\n    logp=F.log_softmax(logits[0].float(),dim=-1)\n    positions=torch.arange(len(prefix)-1,len(ids)-1,device=device)\n    target=torch.tensor(suffix,device=device,dtype=torch.long)\n    return -float(logp[positions,target].mean().item())\n\n\ndef run_copying(model,sp,cfg,device,name,milestone,stage=\'regular\',test=False):\n    model.eval();rng=random.Random(cfg.seed+(871_999 if test else 67_899))\n    values=eligible_values(sp)\n    filler=filler_pieces(sp)\n    gaps=[0,16,32,64,128,256,384]\n    n=4 if stage==\'mini\' else (8 if stage==\'regular\' else (64 if test else 16))\n    rows=[]\n    for gap in gaps:\n        for i in range(n):\n            if len(values)<4 or not filler:\n                rows.append({\'model\':name,\'tokens\':milestone,\'split\':\'test\' if test else \'validation\',\n                             \'gap\':gap,\'example_id\':i,\'nll_true\':None,\'nll_control\':None,\n                             \'copying_benefit_nats\':None,\'status\':\'SKIP_VOCAB\'})\n                continue\n            selected=rng.sample(values,min(8,len(values)))\n            true=[t[1] for t in selected[:4]]\n            wrong=[t[1] for t in selected[4:8]]\n            if len(wrong)!=4:wrong=list(reversed(true))\n            start=sp.encode(\'Earlier words: \',out_type=int)\n            end=sp.encode(\'\\nRepeat the earlier words: \',out_type=int)\n            mid=[filler[j%len(filler)] for j in range(gap)]\n            good=start+true+mid+end\n            bad=start+wrong+mid+end\n            nll_good=score_suffix(model,good,true,device,cfg)\n            nll_bad=score_suffix(model,bad,true,device,cfg)\n            rows.append({\'model\':name,\'tokens\':milestone,\'split\':\'test\' if test else \'validation\',\n                         \'gap\':gap,\'example_id\':i,\'nll_true\':nll_good,\'nll_control\':nll_bad,\n                         \'copying_benefit_nats\':nll_bad-nll_good if nll_good is not None and nll_bad is not None else None,\n                         \'status\':\'OK\' if nll_good is not None and nll_bad is not None else \'SKIP_CONTEXT\'})\n    out=Path(cfg.root)/\'probes\'/(\'test\' if test else \'validation\')\n    for r in rows:append_csv(out/\'copying.csv\',r)\n    usable=[r[\'copying_benefit_nats\'] for r in rows if r[\'copying_benefit_nats\'] is not None]\n    return {\'copy_mean_gain\':float(np.mean(usable)) if usable else None,\'copy_n\':len(usable),\n            \'copy_skipped\':len(rows)-len(usable)}\n\n\ndef ablations(model,sp,cfg,device,name,milestone,test=False):\n    if not name.endswith(\'_pool\'):return None\n    arr=np.memmap(Path(cfg.root)/\'data\'/(\'test.bin\' if test else \'valid.bin\'),dtype=np.uint16,mode=\'r\')\n    # Same held-out token positions across all modes; also same small paired probe set.\n    rows=[]\n    for mode in (\'normal\',\'uniform\',\'shifted\',\'off\'):\n        model.set_pool_mode(mode)\n        metrics=evaluate_nll(model,arr,cfg,device,max_positions=(8192 if test else 4096))\n        result=run_probes(model,sp,cfg,device,name,milestone,stage=\'mini\',test=test,pool_mode=mode)\n        row={\'model\':name,\'tokens\':milestone,\'split\':\'test\' if test else \'validation\',\n             \'mode\':mode,\'nll\':metrics[\'nll\'],\'ppl\':metrics[\'ppl\'],\'mini_probe_accuracy\':result[\'accuracy\']}\n        rows.append(row)\n        append_csv(Path(cfg.root)/\'probes\'/\'pool_ablations.csv\',row)\n    model.set_pool_mode(\'normal\')\n    return rows\n\n\ndef paired_bootstrap(a,b,seed=4,repetitions=1500):\n    # model A minus model B, paired binary outcomes by exact probe IDs.\n    if not a:return None\n    rng=np.random.default_rng(seed)\n    diffs=np.array(a,dtype=np.float64)-np.array(b,dtype=np.float64)\n    idx=rng.integers(0,len(diffs),size=(repetitions,len(diffs)))\n    means=diffs[idx].mean(axis=1)\n    return {\'n\':len(diffs),\'mean_difference\':float(diffs.mean()),\n            \'ci_low_95\':float(np.quantile(means,0.025)),\n            \'ci_high_95\':float(np.quantile(means,0.975))}\n\n\ndef create_paired_summary(cfg):\n    """Paired, per-item comparisons; quantify both pooling and weight sharing."""\n    root=Path(cfg.root)\n    raw=root/\'probes\'/\'test\'/\'final_probe_rows.csv\'\n    if not raw.exists():return None\n    by_model=defaultdict(dict)\n    with raw.open(newline=\'\',encoding=\'utf8\') as f:\n        for row in csv.DictReader(f):\n            if row[\'status\']==\'OK\' and row[\'pool_mode\']==\'normal\' and row[\'stage\']==\'full\':\n                by_model[row[\'model\']][row[\'probe_id\']]=(int(row[\'choice_correct\']),row[\'kind\'],row[\'condition\'],row[\'role\'])\n    comparisons=[(n,\'standard\',\'matrix_vs_transformer\') for n in cfg.models if n!=\'standard\']+[(n,\'matrix_fixed\',\'matrix_ablation\') for n in cfg.models if n not in (\'standard\',\'matrix_fixed\')]\n    report=[]\n    for a,b,description in comparisons:\n        if a not in cfg.models or b not in cfg.models:continue\n        ra,rb=by_model[a],by_model[b]\n        ids=sorted(set(ra)&set(rb))\n        if not ids:continue\n        for kind in [\'ALL\']+sorted(set(ra[k][1] for k in ids)):\n            subset=[k for k in ids if kind==\'ALL\' or ra[k][1]==kind]\n            result=paired_bootstrap([ra[k][0] for k in subset],[rb[k][0] for k in subset],seed=cfg.seed)\n            report.append({\'model_a\':a,\'model_b\':b,\'contrast\':description,\'task\':kind,**result})\n    atomic_json(root/\'reports\'/\'paired_final_bootstrap.json\',report)\n    # Matched test-window NLL differences with a window-level (not token-level) bootstrap.\n    nll_comparisons=[]\n    windows={}\n    for name in cfg.models:\n        p=root/\'reports\'/f\'{name}_test_windows.json\'\n        if p.exists():windows[name]=read_json(p)\n    for a,b,description in comparisons:\n        if a not in windows or b not in windows:continue\n        aa=windows[a];bb=windows[b]\n        n=min(len(aa),len(bb))\n        if n<2:continue\n        # Same test windows and scoring positions in each model.\n        result=paired_bootstrap([aa[i][\'nll\'] for i in range(n)],\n                                [bb[i][\'nll\'] for i in range(n)],seed=cfg.seed+11)\n        nll_comparisons.append({\'model_a\':a,\'model_b\':b,\'contrast\':description,\n                                \'unit\':\'test_window_nll_nats_per_token\',**result})\n    atomic_json(root/\'reports\'/\'paired_test_nll_bootstrap.json\',nll_comparisons)\n    return report\n\n\n# ------------------ Milestone training and evaluation ------------------\n\ndef create_models(cfg,vocab,device,manifest_hash):\n    models={};opts={};scalers={};seen={};steps={}\n    for name in cfg.models:\n        seed_everything(cfg.seed);m=Model(cfg,name,vocab).to(device)\n        opt=torch.optim.AdamW(m.parameters(),lr=cfg.max_lr,betas=(0.9,0.95),eps=1e-8,weight_decay=cfg.weight_decay)\n        scaler=make_scaler(cfg,device);cursor,nstep=try_restore(cfg,name,m,opt,scaler,manifest_hash)\n        models[name]=m;opts[name]=opt;scalers[name]=scaler;seen[name]=cursor;steps[name]=nstep\n    target=param_count(Model(cfg,\'standard\',vocab)); specs={name:{\'parameters\':param_count(m),\'difference_pct\':100*(param_count(m)-target)/target,\'ff_width\':m.ff_width,\'matrix_heads\':cfg.matrix_heads if name!=\'standard\' else None} for name,m in models.items()}\n    assert all(abs(v[\'difference_pct\'])<1 for v in specs.values()),specs\n    atomic_json(Path(cfg.root)/\'model_specs.json\',specs);print(\'PARAMETERS\',specs,flush=True)\n    return models,opts,scalers,seen,steps\n\n\ndef train_one_milestone(cfg,name,model,optimizer,scaler,arr,device,old_seen,old_steps,target,deadline=None):\n    model.train();seen=old_seen;steps=old_steps\n    train_loss=0.;tokens=0;max_grad=0.;last_grad={};\n    if device.type==\'cuda\':torch.cuda.reset_peak_memory_stats()\n    start=time.monotonic();last_lr=None\n    while seen<target:\n        if STOP_REQUESTED or (deadline is not None and time.monotonic()>=deadline):break\n        room=min(target-seen,cfg.batch*cfg.context)\n        b,s=choose_batch(room,cfg)\n        x,y=tensor_from_tokens(arr,seen,b,s,device)\n        lr=learning_rate(seen,cfg)\n        for pg in optimizer.param_groups:pg[\'lr\']=lr\n        optimizer.zero_grad(set_to_none=True)\n        with amp_context(cfg,device):\n            logits=model(x)\n            loss=F.cross_entropy(logits.float().reshape(-1,logits.size(-1)),y.reshape(-1))\n        if not torch.isfinite(loss).item():\n            raise FloatingPointError(\'Non-finite loss: \'+str(name)+\' at scored token \'+str(seen))\n        scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        gn=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip,error_if_nonfinite=True)\n        gn=float(gn.detach().item())\n        if not math.isfinite(gn):raise FloatingPointError(\'Non-finite gradient norm at \'+str(seen))\n        last_grad={\'attention_grad\':model_grad_norm(model,\'.attn.qkv.\') if name==\'standard\' else model_grad_norm(model,\'.attn.kvq.\'),\n                   \'mlp_grad\':model_grad_norm(model,\'shared_ff\' if name.startswith(\'shared\') else \'.ff.\'),\n                   \'decay_grad\':model_grad_norm(model,\'decay_logit\') if name in (\'matrix_learned\',\'matrix_multi\') else None}\n        max_grad=max(max_grad,gn)\n        scaler.step(optimizer);scaler.update()\n        n=b*s;seen+=n;steps+=1;tokens+=n;train_loss+=float(loss.item())*n;last_lr=lr\n    if device.type==\'cuda\':torch.cuda.synchronize()\n    elapsed=time.monotonic()-start\n    return seen,steps,{\'train_nll\':train_loss/max(tokens,1),\'tokens_this_block\':tokens,\n          \'train_elapsed_sec\':elapsed,\'throughput_targets_per_sec\':tokens/max(elapsed,1e-9),\n          \'max_grad_norm\':max_grad,\'lr\':last_lr,\n          \'gpu_peak_allocated_mb\':float(torch.cuda.max_memory_allocated()/1e6) if device.type==\'cuda\' else None,\n          **last_grad}\n\n\ndef evaluate_milestone(cfg,name,model,sp,valid,device,seen,train_metrics):\n    stage= \'full\' if seen%cfg.full_eval_every==0 or seen==cfg.scored_tokens else (\n            \'regular\' if seen%cfg.probe_every==0 else \'none\')\n    n=32768 if stage==\'full\' else 4096\n    for block in model.blocks:block.attn.capture_stats=True\n    val=evaluate_nll(model,valid,cfg,device,max_positions=n)\n    for block in model.blocks:block.attn.capture_stats=False\n    stats=model.pool_stats()\n    row={\'model\':name,\'tokens\':seen,\'milestone_m\':round(seen/1e6,3),\n         \'timestamp_utc\':__import__(\'datetime\').datetime.now(__import__(\'datetime\').timezone.utc).isoformat(),\n         \'parameters\':param_count(model),\'val_nll\':val[\'nll\'],\'val_ppl\':val[\'ppl\'],\n         \'val_token_acc\':val[\'token_accuracy\'],\'prediction_diversity\':val[\'unique_predicted_tokens\'],\n         \'eval_positions\':val[\'positions\'],\'eval_stage\':stage,\n         \'mean_write_gate\':stats.get(\'gate\'),\'pool_entropy_norm\':None,\n         **train_metrics, \'status\':\'OK\'}\n    if not math.isfinite(val[\'nll\']):row[\'status\']=\'FAIL_NONFINITE_VALIDATION\'\n    append_csv(Path(cfg.root)/\'metrics\'/\'health.csv\',row)\n    if stage!=\'none\':\n        try:\n            probe_result=run_probes(model,sp,cfg,device,name,seen,stage=\'full\' if stage==\'full\' else \'regular\')\n            copy_result=run_copying(model,sp,cfg,device,name,seen,\'full\' if stage==\'full\' else \'regular\')\n            m={\'model\':name,\'tokens\':seen,\'stage\':stage,\n               \'probe_accuracy\':probe_result[\'accuracy\'],\n               \'probe_unrestricted_accuracy\':probe_result[\'unrestricted_accuracy\'],\n               \'probe_count\':probe_result[\'count\'],\n               \'probe_skipped\':probe_result[\'skipped\'],**copy_result}\n            append_csv(Path(cfg.root)/\'metrics\'/\'probe_milestones.csv\',m)\n        except Exception as exc:\n            # An optional synthetic probe may fail, but that should not destroy\n            # expensive model training. The failure is NOT reported as accuracy.\n            append_csv(Path(cfg.root)/\'metrics\'/\'evaluation_errors.csv\',\n                       {\'model\':name,\'tokens\':seen,\'stage\':stage,\'part\':\'probes/copy\',\n                        \'error\':repr(exc),\'traceback\':traceback.format_exc()[-12000:]})\n            print(\'NONFATAL PROBE FAILURE\',name,seen,repr(exc),flush=True)\n        finally:\n            model.set_pool_mode(\'normal\')\n    if False:  # legacy pooling diagnostic disabled\n        try:ablations(model,sp,cfg,device,name,seen,test=False)\n        except Exception as exc:\n            append_csv(Path(cfg.root)/\'metrics\'/\'evaluation_errors.csv\',\n                       {\'model\':name,\'tokens\':seen,\'stage\':stage,\'part\':\'pool_ablations\',\n                        \'error\':repr(exc),\'traceback\':traceback.format_exc()[-12000:]})\n            print(\'NONFATAL ABLATION FAILURE\',name,seen,repr(exc),flush=True)\n        finally:model.set_pool_mode(\'normal\')\n    return row\n\n\ndef final_test(cfg,models,sp,device):\n    root=Path(cfg.root);done=root/\'reports\'/\'final_complete.json\'\n    if done.exists():return read_json(done)\n    if not all((root/\'checkpoints\'/name/\'progress.json\').exists() and\n               read_json(root/\'checkpoints\'/name/\'progress.json\')[\'token_cursor\']>=cfg.scored_tokens\n               for name in cfg.models):\n        print(\'Final test is locked until all configured models complete the target.\',flush=True)\n        return None\n    test=np.memmap(root/\'data\'/\'test.bin\',dtype=np.uint16,mode=\'r\')\n    results={}\n    # Untouched test set first opened here; earlier stages only use valid.bin.\n    for name in cfg.models:\n        model=models[name]\n        met=evaluate_nll(model,test,cfg,device,max_positions=min(100_000,len(test)-1))\n        results[name]={k:v for k,v in met.items() if k!=\'window_nll\'}\n        atomic_json(root/\'reports\'/f\'{name}_test_windows.json\',met[\'window_nll\'])\n        pr=run_probes(model,sp,cfg,device,name,cfg.scored_tokens,stage=\'full\',test=True)\n        cp=run_copying(model,sp,cfg,device,name,cfg.scored_tokens,stage=\'full\',test=True)\n        results[name].update({\'probe_accuracy\':pr[\'accuracy\'],\n                              \'probe_unrestricted_accuracy\':pr[\'unrestricted_accuracy\'],\'pair_all_correct\':pr[\'pair_all_correct\'],\'complete_pairs\':pr[\'complete_pairs\'],\n                              **cp})\n        if not cfg.smoke:\n            window_audit(model,test,cfg,device,name,cfg.scored_tokens,\'test\');real_repeat_audit(model,test,cfg,device,name,cfg.scored_tokens,\'test\');long_fact_audit(model,sp,test,cfg,device,name,cfg.scored_tokens,\'test\');backend_audit(model,cfg,device,name,cfg.scored_tokens)\n        print(\'FINAL TEST\',name,\'ppl=\',round(results[name][\'ppl\'],3),\n              \'probes=\',round(pr[\'accuracy\'],4) if pr[\'accuracy\'] is not None else \'NA\',flush=True)\n    atomic_json(root/\'reports\'/\'final_results.json\',results)\n    create_paired_summary(cfg)\n    atomic_json(done,{\'completed\':True,\'tokens_per_model\':cfg.scored_tokens,\n          \'models\':list(cfg.models),\'results_sha256\':sha256_file(root/\'reports\'/\'final_results.json\')})\n    return results\n\n\ndef stop_warnings(cfg,name,current):\n    path=Path(cfg.root)/\'metrics\'/\'health.csv\'\n    if not path.exists():return []\n    with path.open(newline=\'\') as f:rows=[r for r in csv.DictReader(f) if r[\'model\']==name]\n    if len(rows)<3:return []\n    last=rows[-3:]\n    recent=[float(r[\'val_nll\']) for r in last]\n    reasons=[]\n    if recent[1]>recent[0]+0.45 and recent[2]>recent[1]+0.45:\n        reasons.append(\'two_severe_validation_regressions\')\n    if float(last[-1][\'prediction_diversity\'])<3:reasons.append(\'prediction_collapse\')\n    if reasons:append_csv(Path(cfg.root)/\'metrics\'/\'warnings.csv\',\n                          {\'model\':name,\'tokens\':current,\'warning\':\';\'.join(reasons)})\n    return reasons\n\n\ndef generate_readme(cfg):\n    reps=Path(cfg.root)/\'reports\';reps.mkdir(exist_ok=True,parents=True)\n    (reps/\'READ_ME_FIRST.md\').write_text("""# Matrix EXILRM 500M experiment\nReference standard Transformer: learned positions, four attention heads, private FFN. Matrix: original access/write/candidate gates, key/value outer product, strict previous-input exponential history, query read. Four matrix variants: fixed 0.9, learned uniform 0.9, learned multi-timescale, fixed 0.9 with per-head RMSNorm. FF widths match total parameters; this changes FF capacity across architectures. No task training or special input features. FFT/scan are equivalent history backends; nonlinear matrix recurrence remains sequential.\n500M scored targets PER model. Single seed. Data hash-split at document level; exact duplicates grouped, near duplicates not guaranteed removed. Validation probes are synthetic zero-shot tests; final test separate. Multiple-choice chance 25%; interpret paired and unrestricted results too. Full-window audit reports actual context: reference learned positional table cannot extrapolate past training context, so it uses a recent suffix above that limit. Matrix receives the requested full prefix. Compare matched windows separately. Timings implementation-specific. Checkpoints and token cache excluded from report ZIP; preserve experiment root to resume.\n""")\n\n\ndef export_report(cfg):\n    root=Path(cfg.root);root.mkdir(exist_ok=True,parents=True)\n    generate_readme(cfg)\n    dst=Path(\'/kaggle/working/RESEARCH_RESULTS_UNDER_350MB.zip\') if Path(\'/kaggle/working\').exists() else root.parent/\'RESEARCH_RESULTS_UNDER_350MB.zip\'\n    include=[]\n    for fn in (\'experiment_config.json\',\'experiment_manifest.json\',\'model_specs.json\',\'research500.py\'):\n        p=root/fn\n        if p.exists():include.append(p)\n    if Path(__file__).exists() and Path(__file__).name==\'research500.py\' and Path(__file__).resolve()!= (root/\'research500.py\').resolve():\n        # This happens in local smoke tests, not Kaggle. Include source by copying.\n        shutil.copy2(Path(__file__),root/\'research500.py\')\n        include.append(root/\'research500.py\')\n    for sub in (\'metrics\',\'probes\',\'reports\'):\n        folder=root/sub\n        if folder.exists():include.extend(p for p in folder.rglob(\'*\') if p.is_file())\n    for fn in (\'data/manifest.json\',\'data/local_source_manifest.json\',\'data/source_revision.json\',\'data/tokenizer_sample_manifest.json\',\'data/spm.model\',\'data/spm.vocab\'):\n        p=root/fn\n        if p.exists():include.append(p)\n    # Never include .pt, .bin, random text samples or files outside the allowlist.\n    include=sorted(set(include))\n    tmp=dst.with_suffix(\'.zip.part\')\n    with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:\n        for p in include:\n            if p.suffix in (\'.pt\',\'.pth\',\'.bin\',\'.npy\',\'.parquet\'):continue\n            z.write(p,arcname=str(p.relative_to(root)))\n    size=tmp.stat().st_size\n    limit=min(cfg.report_zip_limit_mb,349)*1024*1024\n    if size>=limit:\n        tmp.unlink(missing_ok=True)\n        raise RuntimeError(f\'Report ZIP {size/1e6:.1f}MB would exceed safe {cfg.report_zip_limit_mb}MB; reduce probe counts\')\n    tmp.replace(dst)\n    print(\'DOWNLOADABLE REPORT:\',dst,\'size\',round(size/1e6,2),\'MB,\',len(include),\'candidate files\',flush=True)\n    return dst\n\n\ndef run(cfg):\n    cfg.check()\n    out=Path(cfg.root);out.mkdir(parents=True,exist_ok=True)\n    if cfg.resume_input_root:copy_previous_session(cfg)\n    freeze_config(cfg)\n    session_start=time.monotonic()\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if not cfg.smoke and device.type!=\'cuda\':\n        raise RuntimeError(\'500M-token Kaggle run requires GPU. Enable Settings -> Accelerator -> GPU.\')\n    signal.signal(signal.SIGTERM,request_stop)\n    if hasattr(signal,\'SIGINT\'):signal.signal(signal.SIGINT,request_stop)\n    print(\'EXPERIMENT:\',VERSION,\'device\',device,\'model variants\',cfg.models,flush=True)\n    print(\'Research report ZIP will contain logs/results, NOT 1GB corpus or optimizer checkpoints.\',flush=True)\n    models={}\n    try:\n        manifest=prepare_corpus(cfg)\n        if not manifest.get(\'completed\'):raise RuntimeError(\'Corpus incomplete; resume preparation first\')\n        import sentencepiece as spm\n        sp=spm.SentencePieceProcessor(model_file=str(out/\'data\'/\'spm.model\'))\n        manifest_hash=sha256_file(out/\'data\'/\'manifest.json\')\n        probe_preflight(sp,cfg)\n        if not (out/\'reports\'/\'gradient_check.json\').exists():\n            checks=sanity_gradients(cfg,device,sp.vocab_size())\n            atomic_json(out/\'reports\'/\'gradient_check.json\',checks)\n            print(\'Gradient preflight passed:\',[z[\'model\'] for z in checks],flush=True)\n        models,opts,scalers,seen,steps=create_models(cfg,sp.vocab_size(),device,manifest_hash)\n        valid=np.memmap(out/\'data\'/\'valid.bin\',dtype=np.uint16,mode=\'r\')\n        train=np.memmap(out/\'data\'/\'train.bin\',dtype=np.uint16,mode=\'r\')\n        atomic_json(out/\'experiment_manifest.json\',{\n          \'version\':VERSION,\'dataset_manifest_sha256\':manifest_hash,\n          \'scored_targets_per_model\':cfg.scored_tokens,\n          \'models\':list(cfg.models),\'device\':str(device),\'torch\':torch.__version__,\n          \'numpy\':np.__version__,\'python\':platform.python_version(),\n          \'experimental_design\':\'reference standard Transformer vs four matrix EXILRM variants\',\n          \'training_schedule\':\'same frozen contiguous scored token windows for all models, interleaved by 1M milestones\',\n          \'validation_looked_at_during_training\':True,\'test_only_at_final\':True,\n          \'history_backend\':cfg.history_backend,\'compiled_chunks_requested\':cfg.compile_chunks,\'no_memory_task_training\':True,\n          \'token_counts_include_repeated_training_source_documents\':\'not intentionally repeated, exact dupes may exist in source\'})\n        milestone_blocks_done=0\n        while any(seen[k]<cfg.scored_tokens for k in cfg.models):\n            for name in cfg.models:\n                if seen[name]>=cfg.scored_tokens:continue\n                if STOP_REQUESTED:break\n                if cfg.max_session_minutes and time.monotonic()-session_start > cfg.max_session_minutes*60:\n                    print(\'SESSION TIME BUDGET REACHED; saving complete checkpoint states\',flush=True)\n                    return {\'completed\':False,\'cursors\':seen}\n                model=models[name]\n                target=min(cfg.scored_tokens,(seen[name]//cfg.checkpoint_every+1)*cfg.checkpoint_every)\n                try:\n                    new_seen,new_steps,train_met=train_one_milestone(cfg,name,model,opts[name],scalers[name],train,device,\n                                                                      seen[name],steps[name],target,session_start+cfg.max_session_minutes*60 if cfg.max_session_minutes else None)\n                except (FloatingPointError,RuntimeError) as e:\n                    print(\'TRAINING FAILURE\',name,str(e),flush=True)\n                    append_csv(out/\'metrics\'/\'warnings.csv\',{\'model\':name,\'tokens\':seen[name],\n                               \'warning\':\'TRAINING_EXCEPTION:\'+repr(e)})\n                    raise\n                seen[name]=new_seen;steps[name]=new_steps\n                export_checkpoint(cfg,name,model,opts[name],scalers[name],new_seen,new_steps,manifest_hash,\n                                  reason=\'interrupted\' if STOP_REQUESTED else \'milestone\')\n                if new_seen<target and not STOP_REQUESTED:\n                    return {\'completed\':False,\'cursors\':seen}\n                if new_seen==target:\n                    row=evaluate_milestone(cfg,name,model,sp,valid,device,new_seen,train_met)\n                    flags=stop_warnings(cfg,name,new_seen)\n                    if cfg.stop_on_two_severe_val_regressions and flags:\n                        raise RuntimeError(\'Stopping due to requested validation guardrail: \'+str(flags))\n                    print(\'MILESTONE\',name,f\'{new_seen/1e6:.1f}M\',\n                          \'validation_ppl\',round(row[\'val_ppl\'],3),\n                          \'targets/s\',round(train_met[\'throughput_targets_per_sec\']),\n                          \'write_gate\',round(row[\'mean_write_gate\'],3) if row[\'mean_write_gate\'] is not None else \'-\',\n                          \'warning\',\',\'.join(flags) if flags else \'-\',flush=True)\n                    milestone_blocks_done+=1\n                if (new_seen%5_000_000==0 or STOP_REQUESTED):export_report(cfg)\n                if cfg.max_milestones_per_session and milestone_blocks_done>=cfg.max_milestones_per_session:\n                    print(\'CONFIGURED MILESTONE SESSION LIMIT; resume for the rest\',flush=True)\n                    return {\'completed\':False,\'cursors\':seen}\n                if STOP_REQUESTED:break\n            if STOP_REQUESTED:break\n        complete=all(seen[k]>=cfg.scored_tokens for k in cfg.models)\n        if complete:\n            final_test(cfg,models,sp,device)\n        return {\'completed\':complete,\'cursors\':seen}\n    finally:\n        try:export_report(cfg)\n        except Exception as e:print(\'WARNING report export failed:\',repr(e),flush=True)\n\n\nif __name__==\'__main__\':\n    from argparse import ArgumentParser\n    ap=ArgumentParser()\n    ap.add_argument(\'--smoke\',action=\'store_true\')\n    ap.add_argument(\'--root\',default=None)\n    ap.add_argument(\'--milestones\',type=int,default=4)\n    args=ap.parse_args()\n    if args.smoke:\n        cfg=Config(root=args.root or \'./fineweb500_offline_smoke\',scored_tokens=8192,\n             valid_tokens=8193,test_tokens=8193,tokenizer_sample_chars=50000,\n             vocab=256,context=64,width=32,hidden=64,layers=2,heads=4,\n             batch=2,checkpoint_every=2048,archive_every=4096,probe_every=4096,\n             full_eval_every=8192,max_session_minutes=0,\n             max_milestones_per_session=args.milestones,smoke=True)\n    else:\n        cfg=Config(root=args.root or Config.root)\n    print(\'RUN RESULT:\',run(cfg))', encoding='utf-8')

In [ ]:
(WORK/'exilrm_time_budget.py').write_text('"""Time-budgeted Kaggle runner; uses unmodified research500 model equations."""\nfrom __future__ import annotations\nimport os, sys, time, json, math, signal, subprocess, traceback, zipfile, shutil\nfrom pathlib import Path\nfrom dataclasses import asdict\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nimport research500 as R\n\nMODELS=[\'standard\',\'matrix_fixed\']\nMATRIX_MODES=[\'fft_eager\',\'parallel_eager\',\'scan_eager\',\'fft_graph\',\'parallel_graph\',\'scan_graph\']\ndef mode_parts(name,mode):\n    if name==\'standard\':return \'fft\',mode\n    return tuple(mode.split(\'_\',1))\nWAVES=[MODELS]\n\ndef equal_budget(rates, seconds, safety=.75):\n    # Each wave takes target / its slowest worker rate. Waves run serially.\n    cost=sum(1/min(rates[n] for n in wave) for wave in WAVES)\n    return int(max(0,seconds)*safety/cost),cost\n\n\ndef atomic(p,d): R.atomic_json(p,d)\ndef now(): return time.time()\ndef data_info(root):\n    d=Path(root)/\'data\'; m=R.read_json(d/\'manifest.json\')\n    if not m: raise RuntimeError(\'No prepared corpus manifest. Attach local Parquet or a prepared data directory.\')\n    n=m[\'sizes\'] if m.get(\'completed\') else m[\'filled\']\n    for k in (\'train\',\'valid\',\'test\'):\n        if n[k]<8193 or (d/f\'{k}.bin\').stat().st_size<2*n[k]:raise RuntimeError(\'Insufficient prepared \'+k+\' tokens: \'+str(n))\n    return d,m,n\n\ndef arrays(data_root):\n    d,m,n=data_info(data_root)\n    return {k:np.memmap(d/f\'{k}.bin\',dtype=np.uint16,mode=\'r\',shape=(n[k],)) for k in n},m\n\ndef setup(cfg,name,device,vocab):\n    R.seed_everything(cfg.seed); m=R.Model(cfg,name,vocab).to(device)\n    opt=torch.optim.AdamW(m.parameters(),lr=cfg.max_lr,betas=(.9,.95),weight_decay=cfg.weight_decay)\n    return m,opt\n\ndef sync(device):\n    if device.type==\'cuda\':torch.cuda.synchronize(device)\n\nclass Step:\n    """CUDA graph captures forward/backward only; AdamW and LR are ordinary eager operations.\n    No model parameters change while preparing the graph. Graph replays overwrite static grads.\n    """\n    def __init__(self,m,opt,cfg,device,mode):\n        self.m=m;self.opt=opt;self.cfg=cfg;self.dev=device;self.mode=mode;self.graph=None\n        self.x=torch.zeros((cfg.batch,cfg.context),dtype=torch.long,device=device);self.y=torch.ones_like(self.x)\n        if mode==\'graph\':\n            if device.type!=\'cuda\':raise RuntimeError(\'CUDA graph needs CUDA\')\n            # Warm up FFT plans, attention kernels and autograd on a side stream.\n            stream=torch.cuda.Stream(device=device);stream.wait_stream(torch.cuda.current_stream(device))\n            with torch.cuda.stream(stream):\n                for _ in range(2):\n                    opt.zero_grad(set_to_none=True);self.loss=self.compute();self.loss.backward()\n            torch.cuda.current_stream(device).wait_stream(stream);sync(device)\n            opt.zero_grad(set_to_none=True)\n            self.graph=torch.cuda.CUDAGraph()\n            with torch.cuda.graph(self.graph):\n                self.loss=self.compute();self.loss.backward()\n            self.graph_logits=self.logits\n            # Validate replay vs the original eager loss and gradients before any updates.\n            with torch.no_grad():self.x.random_(0,m.reference.embed.num_embeddings if m.variant==\'standard\' else m.embed.num_embeddings);self.y.copy_(self.x.roll(-1,1))\n            self.graph.replay();sync(device)\n            g=[p.grad.detach().clone() for p in m.parameters()]; gl=float(self.loss.item())\n            # autograd.grad does not replace the graph\'s static gradient buffers.\n            ref=self.compute(); rg=torch.autograd.grad(ref,tuple(m.parameters()))\n            err=max(float((a-b).abs().max()) for a,b in zip(g,rg)); scale=max(1.,max(float(b.abs().max()) for b in rg))\n            if not (math.isfinite(gl) and abs(gl-float(ref))<3e-4 and err/scale<3e-4):raise RuntimeError(f\'Graph verification failed: loss {gl} vs {float(ref)}, grad error {err}\')\n            print(\'CUDA graph gradient check passed\',err/scale,flush=True)\n    def compute(self):\n        logits=self.m(self.x);self.logits=logits\n        return F.cross_entropy(logits.float().reshape(-1,logits.size(-1)),self.y.reshape(-1))\n    def run(self,x,y,lr):\n        self.x.copy_(x);self.y.copy_(y)\n        for p in self.opt.param_groups:p[\'lr\']=lr\n        if self.graph is not None:self.graph.replay();loss=self.loss;self.last_logits=self.graph_logits\n        else:\n            self.opt.zero_grad(set_to_none=True);loss=self.compute();loss.backward();self.last_logits=self.logits\n        gn=torch.nn.utils.clip_grad_norm_(self.m.parameters(),self.cfg.grad_clip,error_if_nonfinite=True)\n        self.opt.step()\n        return float(loss.detach().item())\n\ndef verify_history(device):\n    # Independent literal summation reference; exercise irregular lengths and slow decays.\n    out=[]\n    for T in (1,7,33,128):\n        gen=torch.Generator(device=device).manual_seed(128+T)\n        x=torch.randn(2,T,3,4,4,device=device,dtype=torch.float64,generator=gen,requires_grad=True)\n        lam=torch.tensor([.5,.9,.9999],device=device,dtype=torch.float64,requires_grad=True)\n        rows=[x[:,0]*0]\n        for t in range(1,T):\n            w=lam[None,:]**torch.arange(t-1,-1,-1,device=device,dtype=x.dtype)[:,None]\n            rows.append((x[:,:t]*w[None,:,:,None,None]).sum(1)/w.sum(0)[None,:,None,None])\n        ref=torch.stack(rows,1);gref=torch.autograd.grad(ref.square().sum(),(x,lam),allow_unused=True)\n        for b in (\'fft\',\'scan\',\'parallel\'):\n            y=R.exponential_history(x,lam,b);g=torch.autograd.grad(y.square().sum(),(x,lam),allow_unused=True)\n            err=float((y-ref).abs().max());ge=max(float(((a if a is not None else torch.zeros_like(v))-(c if c is not None else torch.zeros_like(v))).abs().max()) for a,c,v in zip(g,gref,(x,lam)))\n            # Absolute tolerance scaled for long sums and derivative w.r.t. lambda.\n            if err>1e-10 or ge>1e-7:raise RuntimeError(f\'History equivalence failed {b} {T}: {err} {ge}\')\n            out.append(dict(length=T,backend=b,forward_abs=err,gradient_abs=ge))\n    return out\n\ndef compare_step(root,name,cfg,device,vocab,arr,mode):\n    # Two fresh identical models. Reference is eager FFT; candidate is actual selected mode.\n    # Two different real-text batches check replay does not accumulate stale gradients.\n    m,opt=setup(cfg,name,device,vocab);ref,ropt=setup(cfg,name,device,vocab)\n    history,execution=mode_parts(name,mode);m.set_backend(history);ref.set_backend(\'fft\')\n    st=Step(m,opt,cfg,device,execution);rs=Step(ref,ropt,cfg,device,\'eager\');rows=[]\n    for i in range(2):\n        x,y=R.tensor_from_tokens(arr[\'train\'],i*cfg.batch*cfg.context,cfg.batch,cfg.context,device)\n        a=st.run(x,y,cfg.max_lr);b=rs.run(x,y,cfg.max_lr)\n        le=float((st.last_logits-rs.last_logits).detach().abs().max());ge=max(float((p.grad-q.grad).abs().max()) for p,q in zip(m.parameters(),ref.parameters()));pe=max(float((p-q).detach().abs().max()) for p,q in zip(m.parameters(),ref.parameters()))\n        row=dict(step=i+1,loss_abs=abs(a-b),logit_abs=le,clipped_gradient_abs=ge,parameter_abs=pe)\n        rows.append(row)\n        if not all(math.isfinite(v) for v in row.values()) or le>3e-4 or abs(a-b)>3e-5 or ge>3e-4 or pe>3e-5:\n            raise RuntimeError(f\'Candidate failed full model optimiser-step equivalence: {mode} {row}\')\n    return rows\n\ndef worker(args):\n    action,root,name,mode,device_id=args\n    root=Path(root);plan=R.read_json(root/\'plan.json\');cfg=R.Config(**plan[\'cfg\']);data_root=plan[\'data_root\'];device=torch.device(\'cpu\' if device_id==\'cpu\' else \'cuda:0\')\n    torch.set_num_threads(2);torch.set_float32_matmul_precision(\'highest\')\n    if device.type==\'cuda\':torch.cuda.set_device(0)\n    arr,manifest=arrays(data_root);vocab=manifest[\'vocab\'];out=root/name;out.mkdir(exist_ok=True)\n    m,opt=setup(cfg,name,device,vocab);n=cfg.batch*cfg.context\n    if action==\'eval\':return evaluate(root,name,m,cfg,device,arr,plan)\n    history,execution=mode_parts(name,mode);m.set_backend(history)\n    if action==\'bench\':\n        checks=compare_step(root,name,cfg,device,vocab,arr,mode)\n        if name!=\'standard\' and mode==\'fft_eager\':atomic(out/\'history_reference.json\',verify_history(device))\n        atomic(out/f\'verification_{mode}.json\',checks)\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n    step=Step(m,opt,cfg,device,execution)\n    if action==\'bench\':\n        # Disposable model; never contributes to training token counts or checkpoints.\n        def batch_at(i):\n            return R.tensor_from_tokens(arr[\'train\'],(i*n) % (len(arr[\'train\'])-n),cfg.batch,cfg.context,device)\n        for i in range(2):step.run(*batch_at(i),cfg.max_lr)\n        sync(device);t=now();k=0\n        while k<12 and (k<3 or now()-t<30):\n            step.run(*batch_at(k+2),cfg.max_lr);k+=1\n        sync(device);sec=now()-t\n        atomic(out/f\'bench_{mode}.json\',{\'model\':name,\'mode\':mode,\'tokens_per_second\':n*k/sec,\'seconds\':sec,\'steps\':k,\'device\':torch.cuda.get_device_name(0) if device.type==\'cuda\' else \'cpu\',\'parameters\':R.param_count(m),\'batch\':cfg.batch,\'context\':cfg.context})\n        return\n    if not (out/f\'verification_{mode}.json\').is_file():raise RuntimeError(\'Unverified training mode\')\n    # Restart from the exact fresh seeded initialisation; calibration never transfers weights.\n    target=plan[\'target\'];interval=plan[\'interval\'];deadline=plan[\'train_deadline\'];seen=0;steps=0;start=now();last_progress=start;loss_sum=0.;loss_n=0\n    def save():\n        sync(device);p=out/f\'ckpt_{seen:012d}.pt\';R.atomic_torch(p,{\'model\':m.state_dict(),\'optimizer\':opt.state_dict(),\'tokens\':seen,\'steps\':steps,\'cfg\':asdict(cfg),\'mode\':mode,\'data_manifest\':manifest})\n        atomic(out/\'progress.json\',{\'model\':name,\'tokens\':seen,\'steps\':steps,\'target\':target,\'mode\':mode,\'elapsed\':now()-start,\'actual_tokens_per_second\':seen/max(now()-start,1e-9),\'mean_train_nll\':loss_sum/max(loss_n,1)})\n        print(name,\'tokens\',seen,\'target\',target,\'elapsed\',round(now()-start),\'rate\',round(seen/max(now()-start,1)),flush=True)\n    while seen<target and now()<deadline:\n        x,y=R.tensor_from_tokens(arr[\'train\'],seen,cfg.batch,cfg.context,device)\n        loss=step.run(x,y,R.learning_rate(seen,cfg))\n        if not math.isfinite(loss):raise FloatingPointError(\'Nonfinite loss\')\n        seen+=n;steps+=1;loss_sum+=loss*n;loss_n+=n\n        if seen%interval==0 or seen==target:save()\n        elif now()-last_progress>=60:\n            atomic(out/\'progress.json\',{\'model\':name,\'tokens\':seen,\'steps\':steps,\'target\':target,\'mode\':mode,\'elapsed\':now()-start,\'actual_tokens_per_second\':seen/max(now()-start,1e-9),\'mean_train_nll\':loss_sum/max(loss_n,1)})\n            last_progress=now()\n    if seen and not (out/f\'ckpt_{seen:012d}.pt\').exists():save()\n    atomic(out/\'train_done.json\',{\'tokens\':seen,\'target\':target,\'complete\':seen==target})\n\ndef evaluate(root,name,m,cfg,device,arr,plan):\n    import sentencepiece as spm\n    out=root/name;seen=plan[\'common\'];p=out/f\'ckpt_{seen:012d}.pt\';state=torch.load(p,map_location=device,weights_only=False);m.load_state_dict(state[\'model\']);m.eval()\n    cfg.history_backend=mode_parts(name,state[\'mode\'])[0];m.set_backend(cfg.history_backend)\n    cfg.root=str(out);cfg.window_examples=2;cfg.use_amp=False\n    sp=spm.SentencePieceProcessor(model_file=str(Path(plan[\'data_root\'])/\'data\'/\'spm.model\'))\n    metrics={\'model\':name,\'tokens\':seen,\'parameters\':R.param_count(m),\'backend\':state[\'mode\']}\n    metrics[\'validation\']=R.evaluate_nll(m,arr[\'valid\'],cfg,device,max_positions=8192)\n    metrics[\'test\']=R.evaluate_nll(m,arr[\'test\'],cfg,device,max_positions=8192)\n    atomic(out/\'evaluation.json\',metrics)\n    # Priority order: LM, numerical check, then memory. Deadline supervisor may truncate later tasks.\n    if name!=\'standard\':R.backend_audit(m,cfg,device,name,seen)\n    R.window_audit(m,arr[\'test\'],cfg,device,name,seen,\'test\')\n    R.real_repeat_audit(m,arr[\'test\'],cfg,device,name,seen,\'test\')\n    R.long_fact_audit(m,sp,arr[\'test\'],cfg,device,name,seen,\'test\')\n    rows=[];specs=[(\'recall\',0),(\'recall\',128),(\'binding\',4),(\'binding\',8),(\'overwrite\',0),(\'counterfactual\',0),(\'multi_hop\',2),(\'multi_hop\',3)]\n    for kind,condition in specs:\n        for j in range(8):\n            for pr in R.compose_probe(sp,kind,condition,j,cfg.seed+99000009,cfg.context):\n                pred=R.score_one_probe(m,pr,device,cfg);rows.append({\'model\':name,\'tokens\':seen,\'kind\':kind,\'condition\':condition,\'probe_id\':pr[\'probe_id\'],\'role\':pr[\'role\'],**pred})\n        atomic(out/\'probe_results.json\',rows)\n    if name!=\'standard\':\n        metrics[\'decays\']=[b.attn.decay().detach().cpu().tolist() for b in m.blocks]\n        for b in m.blocks:b.attn.capture_stats=True\n        with torch.inference_mode():m(torch.tensor(np.array(arr[\'test\'][:cfg.context],dtype=np.int64),device=device)[None])\n        metrics[\'write_gates\']=[b.attn.last_gate for b in m.blocks]\n    metrics[\'memory_audit_complete\']=True;atomic(out/\'evaluation.json\',metrics)\n\ndef launch(root,action,name,mode,gpu):\n    env=os.environ.copy();env[\'CUDA_VISIBLE_DEVICES\']=str(gpu) if gpu!=\'cpu\' else \'\';env[\'OMP_NUM_THREADS\']=\'2\';env[\'MKL_NUM_THREADS\']=\'2\'\n    log=Path(root)/\'logs\';log.mkdir(exist_ok=True);f=open(log/f\'{name}_{action}_{mode}.log\',\'w\')\n    p=subprocess.Popen([sys.executable,\'-u\',__file__,\'worker\',action,str(root),name,mode,str(gpu)],env=env,stdout=f,stderr=subprocess.STDOUT);f.close();return p\n\ndef wait_jobs(jobs,deadline,root=None,stage=\'\'):\n    last=now()\n    try:\n        while any(p.poll() is None for p in jobs.values()) and now()<deadline:\n            time.sleep(min(2,max(0,deadline-now())))\n            if now()-last>=60:\n                status={k:(\'running\' if p.poll() is None else f\'exit {p.returncode}\') for k,p in jobs.items()}\n                if root and stage.startswith(\'train\'):\n                    status={k:R.read_json(Path(root)/k/\'progress.json\',v) for k,v in status.items()}\n                print(stage,\'progress:\',status,\'seconds left:\',int(deadline-now()),flush=True);last=now()\n    finally:\n        for p in jobs.values():\n            if p.poll() is None:p.terminate()\n        for p in jobs.values():\n            try:p.wait(timeout=3)\n            except subprocess.TimeoutExpired:p.kill();p.wait()\n    return {k:p.returncode for k,p in jobs.items()}\n\ndef prepare(root,cfg,deadline):\n    p=subprocess.Popen([sys.executable,\'-u\',__file__,\'prep\',str(root)],stdout=open(Path(root)/\'prep.log\',\'w\'),stderr=subprocess.STDOUT)\n    while p.poll() is None and now()<deadline:time.sleep(2)\n    if p.poll() is None:\n        p.terminate()\n        try:p.wait(timeout=10)\n        except subprocess.TimeoutExpired:p.kill();p.wait()\n    return p.returncode\n\ndef orchestrate(root,cfg,deadline,existing_data_root=\'\',cap=100_000_000,smoke=False):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    if (root/\'plan.json\').exists():raise RuntimeError(\'Use a NEW output root for each time-budget run (no mixing checkpoints).\')\n    ng=torch.cuda.device_count();visible=os.environ.get(\'CUDA_VISIBLE_DEVICES\',\'\');gpus=[\'cpu\']*2 if smoke else (visible.split(\',\') if visible else list(range(ng)))[:2]\n    if not smoke and ng<2:raise RuntimeError(f\'Need two visible GPUs for this head-to-head run; detected {ng}.\')\n    cfg.models=tuple(MODELS);cfg.use_amp=False;cfg.compile_chunks=False;cfg.scored_tokens=cap;cfg.root=str(root);cfg.check()\n    cfg.valid_tokens=32768 if not smoke else 8193;cfg.test_tokens=cfg.valid_tokens\n    plan={\'cfg\':asdict(cfg),\'data_root\':str(existing_data_root or root),\'wall_deadline\':deadline,\'seed\':cfg.seed,\'models\':MODELS,\'gpu_count\':ng,\'precision\':\'FP32 (same for all models); history audit float64\',\'smoke\':smoke,\'start\':now()}\n    atomic(root/\'plan.json\',plan)\n    if not existing_data_root:\n        prep_deadline=min(deadline-120,now()+(20*60 if not smoke else 60));print(\'Preparing bounded real-text corpus; stop collection after 20 minutes.\',flush=True)\n        prepare(root,cfg,prep_deadline)\n    d,manifest,lengths=data_info(plan[\'data_root\']);\n    if not smoke and manifest.get(\'source\')==\'SYNTHETIC_SMOKE_ONLY\':raise RuntimeError(\'Synthetic smoke corpus cannot be used for research training\')\n    print(\'Prepared token counts\',lengths,flush=True)\n    # First measure safe eager reference, then other exact history algorithms and graph replay.\n    # Matrix candidates use the same GPU; all are timed on the full training batch shape.\n    stages=[(\'eager\',\'fft_eager\'),(\'graph\',\'parallel_eager\'),(None,\'scan_eager\'),(None,\'fft_graph\'),(None,\'parallel_graph\'),(None,\'scan_graph\')]\n    if smoke:stages=[(\'eager\',\'fft_eager\'),(None,\'parallel_eager\'),(None,\'scan_eager\')]\n    for i,(tm,mm) in enumerate(stages):\n        jobs={\'matrix_fixed\':launch(root,\'bench\',\'matrix_fixed\',mm,gpus[1])}\n        if tm:jobs[\'standard\']=launch(root,\'bench\',\'standard\',tm,gpus[0])\n        wait_jobs(jobs,min(deadline-90,now()+(180 if not smoke else 60)),root,f\'calibration {i+1}: {mm}\')\n    if not (root/\'matrix_fixed\'/\'history_reference.json\').exists():raise RuntimeError(\'Reference history verification did not finish; refusing accelerated training\')\n    rates={};modes={};bench={}\n    for n in MODELS:\n        b=[R.read_json(root/n/f\'bench_{mode}.json\') for mode in ((\'eager\',\'graph\') if n==\'standard\' else MATRIX_MODES)];b=[v for v in b if v and v[\'tokens_per_second\']>0];bench[n]=b\n        if not b:raise RuntimeError(\'No successful benchmark for \'+n+\'; inspect \'+str(root/\'logs\'))\n        best=max(b,key=lambda v:v[\'tokens_per_second\']);rates[n]=best[\'tokens_per_second\'];modes[n]=best[\'mode\']\n    # Reserve 20min for frozen tests/report, 5min before hard limit for termination/export.\n    reserve=20*60 if not smoke else 45;train_end=deadline-reserve-(5*60 if not smoke else 5)\n    batch_tokens=cfg.batch*cfg.context\n    seconds=max(0,train_end-now());estimate,wave_cost=equal_budget(rates,seconds)\n    target=min(estimate,lengths[\'train\']-1,cap)//batch_tokens*batch_tokens\n    if smoke:target=min(target,cap//batch_tokens*batch_tokens)\n    if target<batch_tokens:raise RuntimeError(\'Insufficient time/data for one full common batch\')\n    interval=max(batch_tokens,(max(batch_tokens,target//20)//batch_tokens)*batch_tokens)\n    # target also aligned to shared checkpoint intervals; exact budget both models.\n    target=target//interval*interval\n    cfg.scored_tokens=target;cfg.warmup_tokens=max(batch_tokens,min(4_000_000,target//20));plan.update({\'cfg\':asdict(cfg),\'target\':target,\'interval\':interval,\'train_deadline\':train_end,\'benchmarks\':bench,\'chosen_modes\':modes,\'estimated_rates\':rates,\'available_training_seconds\':seconds,\'safety_factor\':.75,\'waves\':WAVES,\'wave_seconds_per_target\':wave_cost,\'data_counts\':lengths,\'dataset_manifest_sha256\':R.sha256_file(d/\'manifest.json\'),\'parameters\':{n:bench[n][0][\'parameters\'] for n in MODELS}})\n    atomic(root/\'plan.json\',plan);print(\'MEASURED targets/s\',rates,\'selected\',modes,\'matched targets/model\',target,\'train seconds\',round(seconds),flush=True)\n    train_status={}\n    wave_costs=[1/min(rates[n] for n in wave) for wave in WAVES]\n    for i,wave in enumerate(WAVES):\n        # Allocate remaining time by measured cost; unused time from wave 1 passes to wave 2.\n        remaining=max(0,train_end-now())\n        wave_end=min(train_end,now()+remaining*wave_costs[i]/sum(wave_costs[i:]))\n        plan[\'train_deadline\']=wave_end;atomic(root/\'plan.json\',plan)\n        print(\'TRAIN WAVE\',i+1,wave,\'seconds\',round(wave_end-now()),flush=True)\n        jobs={n:launch(root,\'train\',n,modes[n],g) for n,g in zip(wave,gpus)}\n        train_status.update(wait_jobs(jobs,min(train_end+5,wave_end+5),root,f\'train wave {i+1}\'))\n    plan[\'train_exit_codes\']=train_status\n    sets=[{int(p.stem.split(\'_\')[1]) for p in (root/n).glob(\'ckpt_*.pt\')} for n in MODELS];common=set.intersection(*sets)\n    if not common:raise RuntimeError(\'No common saved checkpoint. Inspect worker logs; no fair comparison available.\')\n    plan[\'common\']=max(common);plan[\'cursors\']={n:max(s) if s else 0 for n,s in zip(MODELS,sets)};atomic(root/\'plan.json\',plan);print(\'Fair comparison checkpoint:\',plan[\'common\'],\'cursors:\',plan[\'cursors\'],flush=True)\n    eval_status={};eval_end=deadline-(60 if not smoke else 2)\n    for i,wave in enumerate(WAVES):\n        wave_end=now()+max(0,eval_end-now())/(len(WAVES)-i)\n        jobs={n:launch(root,\'eval\',n,\'eager\',g) for n,g in zip(wave,gpus)}\n        eval_status.update(wait_jobs(jobs,wave_end,root,f\'evaluation wave {i+1}\'))\n    plan[\'evaluation_exit_codes\']=eval_status;atomic(root/\'plan.json\',plan)\n    report={\'plan\':plan,\'results\':{n:R.read_json(root/n/\'evaluation.json\') for n in MODELS},\'completed_equal_target\':plan[\'common\']==target,\'elapsed_seconds\':now()-plan[\'start\']}\n    report[\'comparison_ready\']=all(v is not None for v in report[\'results\'].values());report[\'all_memory_audits_complete\']=all(v is not None and v.get(\'memory_audit_complete\',False) for v in report[\'results\'].values());atomic(root/\'summary.json\',report);return report\n\ndef pack(root):\n    root=Path(root);p=root.parent/(root.name+\'_results.zip\')\n    with zipfile.ZipFile(p,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for f in root.rglob(\'*\'):\n            if f.is_file() and f.suffix in (\'.json\',\'.csv\',\'.jsonl\',\'.log\') and f.stat().st_size<20_000_000:z.write(f,str(f.relative_to(root)))\n        plan=R.read_json(root/\'plan.json\',{});common=plan.get(\'common\');d=Path(plan.get(\'data_root\',str(root)))/\'data\'\n        for f in (d/\'spm.model\',d/\'spm.vocab\',d/\'manifest.json\'):\n            if f.is_file():z.write(f,\'shared_data/\'+f.name)\n        if common is not None:\n            for name in MODELS:\n                f=root/name/f\'ckpt_{common:012d}.pt\'\n                if f.is_file():z.write(f,str(f.relative_to(root)))\n        for f in (Path(__file__),Path(R.__file__)):z.write(f,f.name)\n    return p\n\nif __name__==\'__main__\':\n    if sys.argv[1]==\'worker\':worker(sys.argv[2:])\n    elif sys.argv[1]==\'prep\':\n        root=Path(sys.argv[2]);cfg=R.Config(**R.read_json(root/\'plan.json\')[\'cfg\']);signal.signal(signal.SIGTERM,R.request_stop);R.prepare_corpus(cfg)\n', encoding='utf-8')

## 2 · Configuration and data
Attach your FineWeb-Edu Parquet shard(s) as Kaggle Input. Default glob matches the original notebook. If you already have the earlier tokenized corpus, set `EXISTING_DATA_ROOT` to the directory **containing** `data/manifest.json`, `data/spm.model` and `data/{train,valid,test}.bin`; this skips tokenization and can use the saved tokenizer. The runner uses the recorded filled prefixes if preparation is stopped early, and rejects missing or too-short train/validation/test splits. For a fresh corpus, the document-hash split is preserved; only train documents are used to learn the tokenizer. The fresh tokenizer sample is reduced to 5M characters for time, so reuse the old corpus for exact tokenizer/data consistency with past runs. Do not set the glob to unrelated Parquet files.

In [ ]:
sys.path.insert(0,str(WORK))
import py_compile
py_compile.compile(str(WORK/'research500.py'),doraise=True)
py_compile.compile(str(WORK/'exilrm_time_budget.py'),doraise=True)
from research500 import Config, local_parquet_paths
from exilrm_time_budget import MODELS, orchestrate, pack
# ---- Edit these settings ----
PARQUET_GLOB = '/kaggle/input/**/000_00000.parquet'
EXISTING_DATA_ROOT = ''  # e.g. '/kaggle/input/previous-output/exilrm_500m_experiment'
CAP_TOKENS_PER_MODEL = 100_000_000  # maximum, NOT a promised count
CONTEXT = 256
BATCH = 16
SEED = 20261109
# Use a unique root for each run to prevent mixed checkpoints.
ROOT = WORK / f'exilrm_t4x2_head_to_head_240min_{SEED}_{int(RUN_STARTED)}'
CFG = Config(root=str(ROOT),local_parquet_glob=PARQUET_GLOB,
    dataset_id='HuggingFaceFW/fineweb-edu',dataset_subset='sample-10BT',
    tokenizer_sample_chars=5_000_000,vocab=4096,width=64,hidden=256,layers=3,
    heads=4,matrix_heads=8,context=CONTEXT,batch=BATCH,history_backend='fft',
    compile_chunks=False,use_amp=False,seed=SEED,models=tuple(MODELS),
    max_lr=0.00075,min_lr=0.00005,weight_decay=0.01,grad_clip=1.0,
    stop_on_two_severe_val_regressions=False)
CFG.check()
if EXISTING_DATA_ROOT:
    assert (Path(EXISTING_DATA_ROOT)/'data'/'manifest.json').is_file(), 'Existing root must contain data/manifest.json'
else:
    print('Local parquet files:',[str(x) for x in local_parquet_paths(CFG)])
print('Models:',MODELS,'Context:',CONTEXT,'Batch:',BATCH,'Targets/update:',CONTEXT*BATCH)
print('Output:',ROOT,'Minutes remaining:',round((DEADLINE-time.time())/60,1))

## 3 · Calibrate, train and evaluate

Each candidate starts with the exact same seeded weights and real training batches. The float64 history audit uses direct weighted sums, including odd lengths and decay 0.9999. Each candidate's two successive optimiser steps are checked against eager FFT (standard Transformer uses its own eager reference). Numerical tolerances are logged and enforced; CUDA graph checks also compare raw parameter gradients before clipping. Failed or timed-out candidates are not selected. Measured end-to-end training rates include loss, backward, gradient clipping and AdamW. Calibration weights are discarded. Full training starts afresh and uses the selected mode. All models use FP32. The slowest measured worker and remaining time determine the exact common token budget. Final held-out loss and memory tests use the same checkpoint count; trained EXILRM is evaluated with all three history implementations. Transformer evaluation beyond the learned position table is limited to its recent 256-token context.


In [ ]:
# The deadline includes setup elapsed above. Outputs are packaged even on a caught error.
result = None
try:
    result = orchestrate(ROOT, CFG, DEADLINE, existing_data_root=EXISTING_DATA_ROOT,
                         cap=CAP_TOKENS_PER_MODEL)
except Exception as exc:
    print('RUN STOPPED:',repr(exc))
    print('Inspect ROOT/logs and ROOT/prep.log. Partial outputs will be saved; no incomplete comparison is claimed.')
    import traceback; traceback.print_exc()
finally:
    if ROOT.exists():
        archive=pack(ROOT)
        print('Results archive:',archive)
print('Total wall minutes:',round((time.time()-RUN_STARTED)/60,2))

## 4 · Measured speed and research results
`plan.json` records measured all history/execution candidate rates, selected backend, matched budget, actual common checkpoint, parameters and GPU names. `summary.json` records held-out loss/perplexity and completion flags. **A missing result means that worker failed or was stopped; do not substitute zero.** Backend speedups are measured on the same GPU at the same batch shape, not compared to an unrelated earlier run. Total model-token count = two × common checkpoint. This is the amount of scored training consumed by the two checkpoints used for comparison; physical work may exceed it if faster workers got farther. The ZIP contains logs, metrics, code, tokenizer, manifest and the common model/optimizer checkpoints (not the large token corpus). Other saved checkpoints and the prepared data remain under the run root.

In [ ]:
import json
from IPython.display import display, FileLink
plan_file=ROOT/'plan.json'
if plan_file.exists():
    plan=json.loads(plan_file.read_text())
    print('Selected equal target/model:',plan.get('target'))
    print('Latest shared checkpoint/model:',plan.get('common'))
    if plan.get('common') is not None:
        print('Total tokens in matched comparison:',2*plan['common'])
    print('Actual saved cursors:',plan.get('cursors'))
    rows=[]
    for name, benches in plan.get('benchmarks',{}).items():
        speeds={v['mode']:v['tokens_per_second'] for v in benches}
        rows.append({'model':name,'device':benches[0]['device'],
            **{k+' tok/s':round(v) for k,v in speeds.items()},
            'speedup vs eager reference':round(max(speeds.values())/speeds.get('fft_eager',speeds.get('eager',max(speeds.values()))),2),
            'chosen':plan['chosen_modes'][name],'params':plan['parameters'][name]})
    if rows:
        import pandas as pd
        display(pd.DataFrame(rows))
summary_file=ROOT/'summary.json'
if summary_file.exists():
    summary=json.loads(summary_file.read_text())
    print('Matched checkpoint comparison ready:',summary.get('comparison_ready'))
    print('All memory audits completed:',summary.get('all_memory_audits_complete'))
    rows=[]
    for name, r in summary['results'].items():
        if r is not None:
            rows.append({'model':name,'tokens':r['tokens'],'validation NLL':r['validation']['nll'],
                'validation PPL':r['validation']['ppl'],'test NLL':r['test']['nll'],
                'test PPL':r['test']['ppl'],'memory audit complete':r.get('memory_audit_complete',False)})
    if rows:
        import pandas as pd
        display(pd.DataFrame(rows))
archive=ROOT.parent/(ROOT.name+'_results.zip')
if archive.exists(): display(FileLink(str(archive)))